# 00 · PyTorch essentials, as DETR uses them

> **Code:** every example below is a real line from [`models/`](../models) or [`util/`](../util)

This is not a general PyTorch course. It is the *specific* subset of PyTorch that DETR leans on — the operations which, once they are second nature, turn [`models/detr.py`](../models/detr.py) from hieroglyphics into prose.

Use it two ways:

- **New to PyTorch?** Read it straight through — it assumes nothing, and it is the only notebook with no prerequisites.
- **Already comfortable?** Skip to `01` and come back whenever a line of the tutorial is in your way; the map below is built for exactly that.

Nothing here downloads detection weights. The whole notebook runs in a few seconds on CPU.

---

### The map: operation → the line that needs it

| Operation | Where DETR uses it | § |
|---|---|---|
| `shape` / `dtype` / `device` | [`util/misc.py:321`](../util/misc.py#L321) — building the padded batch | 1 |
| in-place `copy_`, `requires_grad_` | [`util/misc.py:325`](../util/misc.py#L325) — filling the padded buffer | 1 |
| `.flatten(2)` + `.permute()` | [`transformer.py:50`](../models/transformer.py#L50) — grid → sequence | 2 |
| `.view()` vs `.reshape()` | [`transformer.py:59`](../models/transformer.py#L59) — sequence → grid | 2 |
| `stack` / `cat` / `unbind` | [`box_ops.py:10-13`](../util/box_ops.py#L10-L13) — box format conversion | 2 |
| `None` indexing + broadcasting | [`box_ops.py:28`](../util/box_ops.py#L28) — all-pairs IoU | 3 |
| `.repeat()` vs `.expand()` | [`transformer.py:52`](../models/transformer.py#L52) — queries across the batch | 3 |
| `...` and negative indices | [`detr.py:69`](../models/detr.py#L69), [`275`](../models/detr.py#L275) — two very different `-1`s | 4 |
| advanced (tensor) indexing | [`detr.py:119`](../models/detr.py#L119) — scattering matched targets | 4 |
| `.max(-1)` returning **two** things | [`detr.py:275`](../models/detr.py#L275) — logits → labels | 5 |
| `.cumsum()` | [`position_encoding.py:33`](../models/position_encoding.py#L33) — mask-aware coordinates | 5 |
| `@` and `.transpose(-2, -1)` | attention itself — see notebook `01` | 6 |
| `Linear` / `Conv2d(1x1)` / `Embedding` | [`detr.py:37-40`](../models/detr.py#L37-L40) — all four heads | 7 |
| `nn.MultiheadAttention` shapes | [`transformer.py:155`](../models/transformer.py#L155) — `(L, B, E)`, not batch-first | 7 |
| `nn.Module` registration | [`detr.py:33-42`](../models/detr.py#L33-L42) — the five children | 8 |
| `nn.ModuleList` | [`transformer.py:273`](../models/transformer.py#L273) — six stacked layers | 8 |
| `register_buffer` | [`backbone.py:30`](../models/backbone.py#L30) — frozen BN statistics | 8 |
| `requires_grad_(False)` | [`backbone.py:64`](../models/backbone.py#L64) — freezing the backbone | 9 |
| `torch.no_grad()` | [`detr.py:260`](../models/detr.py#L260), [`matcher.py:34`](../models/matcher.py#L34) | 9 |
| `F.interpolate` + dtype dance | [`backbone.py:78`](../models/backbone.py#L78) — downsampling the mask | 10 |
| `F.cross_entropy` on **logits** | [`detr.py:121`](../models/detr.py#L121) — the classification loss | 11 |
| `reduction='none'` | [`detr.py:153`](../models/detr.py#L153) — normalize by box count | 11 |
| `state_dict` / `load_state_dict` | [`main.py:174`](../main.py#L174) — loading the checkpoint | 12 |

In [1]:
import math

import torch
import torch.nn.functional as F
from torch import nn

from detr_utils import *   # noqa: F403  -- puts the repo root on sys.path

torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False, linewidth=110)
print("torch", torch.__version__)

torch 2.14.0


---

## 1. A tensor is three things

Almost every bug you will hit in this tutorial is one of these three being wrong: **shape**, **dtype**, or **device**. Get in the habit of printing all three.

In [2]:
img  = torch.randn(3, 800, 1066)                      # a preprocessed image
mask = torch.ones(1, 800, 1066, dtype=torch.bool)     # util/misc.py:322
ids  = torch.tensor([17, 18], dtype=torch.int64)      # COCO labels: cat, dog

for name, t in [("img", img), ("mask", mask), ("ids", ids)]:
    print(f"{name:5} shape={str(tuple(t.shape)):<18} dtype={str(t.dtype):<14} "
          f"device={t.device}  ndim={t.ndim}  numel={t.numel():,}")

img   shape=(3, 800, 1066)     dtype=torch.float32  device=cpu  ndim=3  numel=2,558,400
mask  shape=(1, 800, 1066)     dtype=torch.bool     device=cpu  ndim=3  numel=852,800
ids   shape=(2,)               dtype=torch.int64    device=cpu  ndim=1  numel=2


**dtype is not cosmetic.** Three dtypes appear in DETR and each is load-bearing:

| dtype | used for | if you get it wrong |
|---|---|---|
| `float32` | images, features, logits | everything silently upcasts; slow |
| `bool` | padding masks | `~mask` becomes bitwise-NOT on ints: `~1 == -2` |
| `int64` | class labels | `F.cross_entropy` refuses to run |

That middle row is worth a demonstration, because the failure is completely silent:

In [3]:
bool_mask = torch.tensor([True, False, True])
int_mask  = torch.tensor([1, 0, 1])

print("~bool_mask :", ~bool_mask, "  <- logical not, what you want")
print("~int_mask  :", ~int_mask,  "  <- bitwise not on int64. Not a mask any more.")
print()
print("Summing them tells the same story:")
print("  (~bool_mask).sum() =", (~bool_mask).sum().item(), " (counts real pixels)")
print("  (~int_mask ).sum() =", (~int_mask).sum().item(),  " (meaningless)")

~bool_mask : tensor([False,  True, False])   <- logical not, what you want
~int_mask  : tensor([-2, -1, -2])   <- bitwise not on int64. Not a mask any more.

Summing them tells the same story:
  (~bool_mask).sum() = 1  (counts real pixels)
  (~int_mask ).sum() = -5  (meaningless)


### Making tensors: the `_like` family

DETR almost never hard-codes a dtype or device when it can copy one. `torch.zeros_like(x)`, `torch.full_like(x, v)` and friends inherit **shape, dtype and device** from `x` — which is why the code works unchanged on CPU, CUDA and MPS.

In [4]:
ref = torch.randn(2, 3, dtype=torch.float64)

print("zeros_like :", torch.zeros_like(ref).dtype, tuple(torch.zeros_like(ref).shape))
print("full_like  :", torch.full_like(ref, 7).flatten()[:3].tolist())
print()
print("transformer.py:55  tgt = torch.zeros_like(query_embed)")
print("  -> the decoder starts from all-zero queries; the LEARNED part is")
print("     query_pos, added inside attention. See notebook 03.")
print()
print("detr.py:195  torch.full_like(src, i)")
print("  -> builds the 'which image' index with src's dtype (int64) for free.")

zeros_like : torch.float64 (2, 3)
full_like  : [7.0, 7.0, 7.0]

transformer.py:55  tgt = torch.zeros_like(query_embed)
  -> the decoder starts from all-zero queries; the LEARNED part is
     query_pos, added inside attention. See notebook 03.

detr.py:195  torch.full_like(src, i)
  -> builds the 'which image' index with src's dtype (int64) for free.


### Here is the padded batch, built from scratch

`nested_tensor_from_tensor_list` ([`util/misc.py:318-327`](../util/misc.py#L318-L327)) is where all three properties come together:

In [5]:
imgs = [torch.randn(3, 800, 1066), torch.randn(3, 800, 800)]   # different sizes!

b, c = len(imgs), 3
h = max(i.shape[1] for i in imgs)
w = max(i.shape[2] for i in imgs)

tensor = torch.zeros((b, c, h, w), dtype=imgs[0].dtype)    # zeros = the padding
mask   = torch.ones((b, h, w), dtype=torch.bool)           # True = padding, EVERYWHERE

for img, pad_img, m in zip(imgs, tensor, mask):
    pad_img[: img.shape[0], : img.shape[1], : img.shape[2]].copy_(img)   # write into a slice
    m[: img.shape[1], : img.shape[2]] = False                            # carve out the real part

print("tensor:", tuple(tensor.shape), tensor.dtype)
print("mask  :", tuple(mask.shape), mask.dtype)
for i in range(b):
    print(f"  image {i}: {(~mask[i]).sum().item():,} real / {mask[i].numel():,} slots "
          f"({100 * (~mask[i]).sum().item() / mask[i].numel():.0f}% real)")

tensor: (2, 3, 800, 1066) torch.float32
mask  : (2, 800, 1066) torch.bool
  image 0: 852,800 real / 852,800 slots (100% real)
  image 1: 640,000 real / 852,800 slots (75% real)


Two things to notice.

**`.copy_(img)`** — a trailing underscore is PyTorch's universal mark for *in-place*. `pad_img[...]` is a **view** into `tensor`, so writing into the view writes into the big tensor. Compare `x.add(1)` (returns a new tensor) with `x.add_(1)` (mutates `x`). This is how you fill a pre-allocated buffer.

**The mask starts all-`True`.** It is easier to allocate "everything is padding" and then carve out the real region. Get this polarity backwards and attention will look only at the grey padding.

In [ ]:
# In-place vs not -- the distinction that makes views dangerous and useful.
base = torch.zeros(4)
view = base[:2]

view.add_(5)        # in-place on a view -> writes through
print("after view.add_(5) , base =", base)

view = view.add(5)  # not in-place -> new tensor, base untouched
print("after view.add(5)  , base =", base)
print()
print("util/misc.py:325 relies on write-through. If .copy_ returned a new")
print("tensor instead, the padded batch would come out all zeros.")

after view.add_(5) , base = tensor([5., 5., 0., 0.])
after view.add(5)  , base = tensor([5., 5., 0., 0.])

util/misc.py:325 relies on write-through. If .copy_ returned a new
tensor instead, the padded batch would come out all zeros.


In [7]:
# The dtype rule that bites people: cross_entropy demands int64 targets.
logits = torch.randn(4, 92)                       # 4 queries, 92 class scores

for target in [torch.tensor([17, 18, 91, 91]),                      # int64  (default)
               torch.tensor([17., 18., 91., 91.])]:                 # float32
    try:
        loss = F.cross_entropy(logits, target)
        print(f"target dtype {str(target.dtype):<14} -> loss = {loss.item():.4f}")
    except RuntimeError as e:
        print(f"target dtype {str(target.dtype):<14} -> RuntimeError: {str(e).splitlines()[0][:58]}")

target dtype torch.int64    -> loss = 5.1296
target dtype torch.float32  -> RuntimeError: expected target dtype to be Long or Byte, but got Float


### device: the one asymmetry to memorize

| | returns | mutates? |
|---|---|---|
| `tensor.to(device)` | a **new** tensor | no — `x.to("cuda")` on its own does nothing |
| `module.to(device)` | `self` | **yes** — moves parameters and buffers in place |

So `model.to(device)` alone is correct, but `x.to(device)` alone is a bug; you need `x = x.to(device)`. That asymmetry is why [`detr_utils.py`](detr_utils.py) writes `model.to(device)` but `NestedTensor.to()` ([`util/misc.py:289`](../util/misc.py#L289)) *returns* a new `NestedTensor`.

In [8]:
x = torch.randn(3)
x.to(torch.float64)                 # <- result thrown away, x unchanged
print("after bare  x.to(float64) :", x.dtype, " <- silently did nothing")
x = x.to(torch.float64)             # <- correct
print("after x = x.to(float64)   :", x.dtype)

m = nn.Linear(3, 3)
print("\nmodule.to() returns self  :", m.to(torch.float64) is m)
print("module param dtype now    :", m.weight.dtype, " <- mutated in place")

after bare  x.to(float64) : torch.float32  <- silently did nothing
after x = x.to(float64)   : torch.float64

module.to() returns self  : True
module param dtype now    : torch.float64  <- mutated in place


---

## 2. Shape surgery: four verbs

If you learn one line from this notebook, learn this one ([`transformer.py:49-50`](../models/transformer.py#L49-L50)):

```python
bs, c, h, w = src.shape
src = src.flatten(2).permute(2, 0, 1)      # (B, C, H, W) -> (HW, B, C)
```

That is the entire bridge from *convolution world* (channels-first grids) to *transformer world* (sequence-first tokens). Let's take it apart.

In [9]:
B, C, H, W = 1, 256, 25, 34                # the real DETR numbers for an 800x1066 image
src = torch.randn(B, C, H, W)

step1 = src.flatten(2)                     # merge dims 2..end  -> (B, C, H*W)
step2 = step1.permute(2, 0, 1)             # reorder axes       -> (H*W, B, C)

print("start            ", tuple(src.shape),   "  (B, C, H, W)     conv world")
print("  .flatten(2)    ", tuple(step1.shape), "  (B, C, HW)       grid is now one axis")
print("  .permute(2,0,1)", tuple(step2.shape), "  (HW, B, C)       sequence world")
print()
print(f"-> {H}x{W} = {H*W} spatial positions became {H*W} tokens of {C} features each.")
print("-> Reading order is row-major: token 34 is the start of the second row.")

start             (1, 256, 25, 34)   (B, C, H, W)     conv world
  .flatten(2)     (1, 256, 850)   (B, C, HW)       grid is now one axis
  .permute(2,0,1) (850, 1, 256)   (HW, B, C)       sequence world

-> 25x34 = 850 spatial positions became 850 tokens of 256 features each.
-> Reading order is row-major: token 34 is the start of the second row.


**`flatten(start_dim)`** merges every dimension from `start_dim` onward. `flatten(2)` on `(B,C,H,W)` leaves `B` and `C` alone and fuses `H,W`. [`transformer.py:53`](../models/transformer.py#L53) does the same to the mask: `mask.flatten(1)` turns `(B,H,W)` into `(B,HW)` so it lines up with the token axis.

**`permute(*dims)`** reorders axes by saying where each new axis comes from. Read `permute(2,0,1)` as: *new axis 0 is old axis 2; new axis 1 is old axis 0; new axis 2 is old axis 1.* `transpose(a,b)` is the two-axis special case — [`transformer.py:59`](../models/transformer.py#L59) uses `hs.transpose(1, 2)`.

**`view` / `reshape`** reinterpret the same numbers under a new shape. `-1` means "infer this one".

In [10]:
t = torch.arange(24).reshape(2, 3, 4)
print("arange(24).reshape(2,3,4):", tuple(t.shape))
print("  .flatten(1)     ->", tuple(t.flatten(1).shape),    " merge last two")
print("  .flatten()      ->", tuple(t.flatten().shape),     " merge everything")
print("  .view(2, -1)    ->", tuple(t.view(2, -1).shape),   " -1 inferred as 12")
print("  .transpose(0,1) ->", tuple(t.transpose(0, 1).shape))
print("  .permute(2,0,1) ->", tuple(t.permute(2, 0, 1).shape))
print()
print("permute is NOT reshape. Same shape, completely different numbers:")
x = torch.arange(6).reshape(2, 3)
print("  x            =", x.tolist())
print("  x.reshape(3,2)=", x.reshape(3, 2).tolist(), " <- reads in memory order")
print("  x.transpose(0,1)=", x.transpose(0, 1).tolist(), " <- actually transposed")

arange(24).reshape(2,3,4): (2, 3, 4)
  .flatten(1)     -> (2, 12)  merge last two
  .flatten()      -> (24,)  merge everything
  .view(2, -1)    -> (2, 12)  -1 inferred as 12
  .transpose(0,1) -> (3, 2, 4)
  .permute(2,0,1) -> (4, 2, 3)

permute is NOT reshape. Same shape, completely different numbers:
  x            = [[0, 1, 2], [3, 4, 5]]
  x.reshape(3,2)= [[0, 1], [2, 3], [4, 5]]  <- reads in memory order
  x.transpose(0,1)= [[0, 3], [1, 4], [2, 5]]  <- actually transposed


### The contiguity trap

`permute` and `transpose` do **not** move data. They return a view with reshuffled *strides* — a different reading order over the same memory. That makes them free, and it makes `view` fragile afterwards.

In [11]:
x = torch.randn(2, 3, 4)
p = x.permute(2, 1, 0)

print("original contiguous?", x.is_contiguous())
print("permuted contiguous?", p.is_contiguous(), " <- same storage, new strides")
print("strides:", x.stride(), "->", p.stride())
print("same storage?", x.untyped_storage().data_ptr() == p.untyped_storage().data_ptr())
print()

try:
    p.view(-1)
except RuntimeError as e:
    print("p.view(-1)    -> RuntimeError:", str(e).splitlines()[0][:68])
print("p.reshape(-1) ->", tuple(p.reshape(-1).shape), " (falls back to a copy)")
print("p.contiguous().view(-1) ->", tuple(p.contiguous().view(-1).shape))

original contiguous? True
permuted contiguous? False  <- same storage, new strides
strides: (12, 4, 1) -> (1, 4, 12)
same storage? True

p.view(-1)    -> RuntimeError: view size is not compatible with input tensor's size and stride (at 
p.reshape(-1) -> (24,)  (falls back to a copy)
p.contiguous().view(-1) -> (24,)


**Rule of thumb:** `reshape` always works; `view` works only when the new shape is compatible with the existing strides, and is guaranteed never to copy. Use `view` when you want a loud failure if a copy would be needed, `reshape` when you just want the shape.

But note the subtlety — **`view` is not about contiguity, it is about stride compatibility.** DETR's return path proves it ([`transformer.py:59`](../models/transformer.py#L59)):

```python
memory.permute(1, 2, 0).view(bs, c, h, w)
```

That is a `view` on a permuted, non-contiguous tensor, and it succeeds, because splitting the last axis `HW` into `H, W` needs no re-striding:

In [12]:
memory = torch.randn(850, B, C)                  # (HW, B, C) -- encoder output
back = memory.permute(1, 2, 0)                   # (B, C, HW)

print("permuted contiguous?", back.is_contiguous(), " ... and yet:")
print("  .view(B, C, 25, 34) ->", tuple(back.view(B, C, 25, 34).shape), " works")
print()
print("Splitting the LAST axis is stride-compatible.")
print("Merging across a permuted boundary is not. That is the whole rule.")

permuted contiguous? False  ... and yet:
  .view(B, C, 25, 34) -> (1, 256, 25, 34)  works

Splitting the LAST axis is stride-compatible.
Merging across a permuted boundary is not. That is the whole rule.


### `stack` vs `cat` vs `unbind`

- `torch.cat(list, dim)` — glue along an **existing** axis. Shapes must match except on `dim`.
- `torch.stack(list, dim)` — create a **new** axis. All shapes must match exactly.
- `x.unbind(dim)` — the inverse of `stack`: remove an axis, return a tuple of slices.

[`box_ops.py:9-13`](../util/box_ops.py#L9-L13) uses `unbind` then `stack` in five lines, and the result is readable in a way that `[..., 0:1]` slicing never is:

In [13]:
from util.box_ops import box_cxcywh_to_xyxy

boxes = torch.tensor([[0.5, 0.5, 0.2, 0.4],       # (cx, cy, w, h), normalized
                      [0.1, 0.9, 0.2, 0.2]])

x_c, y_c, w_, h_ = boxes.unbind(-1)               # split the last axis into 4 named tensors
print("unbind(-1) gave 4 tensors of shape", tuple(x_c.shape), ":", [round(v, 2) for v in x_c.tolist()])

b = [(x_c - 0.5*w_), (y_c - 0.5*h_), (x_c + 0.5*w_), (y_c + 0.5*h_)]
print("\nstack(b, dim=-1):", tuple(torch.stack(b, dim=-1).shape), " <- NEW axis of size 4")
print("cat  (4x (2,1)) :", tuple(torch.cat([t[:, None] for t in b], dim=-1).shape),
      " <- same result, more work")
print("\nmatches the repo:", torch.allclose(torch.stack(b, -1), box_cxcywh_to_xyxy(boxes)))

unbind(-1) gave 4 tensors of shape (2,) : [0.5, 0.1]

stack(b, dim=-1): (2, 4)  <- NEW axis of size 4
cat  (4x (2,1)) : (2, 4)  <- same result, more work

matches the repo: True


In [14]:
# cat vs stack, the one-liner that clarifies it forever
a, bb = torch.zeros(2, 3), torch.ones(2, 3)
print("cat  (dim=0):", tuple(torch.cat([a, bb], 0).shape),   " 2+2 = 4 rows")
print("stack(dim=0):", tuple(torch.stack([a, bb], 0).shape), " a new axis of size 2")
print()
print("DETR uses cat when gathering variable-length per-image data:")
print("  detr.py:116  torch.cat([t['labels'][J] for t, (_, J) in zip(targets, indices)])")
print("  -> image 0 has 2 objects, image 1 has 1 -> a flat tensor of 3 labels.")
print("Stacking would be impossible there: the shapes differ.")

cat  (dim=0): (4, 3)  2+2 = 4 rows
stack(dim=0): (2, 2, 3)  a new axis of size 2

DETR uses cat when gathering variable-length per-image data:
  detr.py:116  torch.cat([t['labels'][J] for t, (_, J) in zip(targets, indices)])
  -> image 0 has 2 objects, image 1 has 1 -> a flat tensor of 3 labels.
Stacking would be impossible there: the shapes differ.


---

## 3. `None`, broadcasting, and the all-pairs trick

**Broadcasting** lets tensors of different shapes combine. The rule, applied right to left:

1. Align shapes from the **right**.
2. A dimension of size **1** stretches to match.
3. A **missing** dimension counts as 1.
4. Anything else is an error.

In [15]:
a = torch.ones(3, 1)
b = torch.ones(   4)       # treated as (1, 4)
print("(3,1) + (4,) ->", tuple((a + b).shape), " <- both stretched")

# indexing with None inserts an axis of size 1 (same as unsqueeze)
v = torch.randn(5)
print("\nv                 ", tuple(v.shape))
print("v[:, None]        ", tuple(v[:, None].shape),      "== v.unsqueeze(1)")
print("v[None, :]        ", tuple(v[None, :].shape),      "== v.unsqueeze(0)")
print("v[:, None, None]  ", tuple(v[:, None, None].shape))
print("v[None]           ", tuple(v[None].shape),         "  (trailing ':' implied)")

(3,1) + (4,) -> (3, 4)  <- both stretched

v                  (5,)
v[:, None]         (5, 1) == v.unsqueeze(1)
v[None, :]         (1, 5) == v.unsqueeze(0)
v[:, None, None]   (5, 1, 1)
v[None]            (1, 5)   (trailing ':' implied)


### Why DETR cares: every pair, no loop

The Hungarian matcher needs the IoU of **every** predicted box against **every** target box. A double `for` loop over 100 × N boxes per image would be unusable. [`box_ops.py:28-32`](../util/box_ops.py#L28-L32) does it with one `None`:

```python
lt = torch.max(boxes1[:, None, :2], boxes2[:, :2])   # (N,1,2) vs (M,2) -> (N,M,2)
rb = torch.min(boxes1[:, None, 2:], boxes2[:, 2:])
wh = (rb - lt).clamp(min=0)
inter = wh[:, :, 0] * wh[:, :, 1]                    # (N,M)
```

In [16]:
from util.box_ops import box_iou

boxes1 = torch.tensor([[0., 0., 10., 10.],       # N = 3 predictions
                       [5., 5., 15., 15.],
                       [20., 20., 30., 30.]])
boxes2 = torch.tensor([[0., 0., 10., 10.],       # M = 2 targets
                       [8., 8., 18., 18.]])

lt = torch.max(boxes1[:, None, :2], boxes2[:, :2])
print("boxes1[:, None, :2] :", tuple(boxes1[:, None, :2].shape))
print("boxes2[:, :2]       :", tuple(boxes2[:, :2].shape))
print("broadcast result    :", tuple(lt.shape), " <- every (pred, target) pair")
print()

iou, union = box_iou(boxes1, boxes2)
print("IoU matrix", tuple(iou.shape), "-- rows = predictions, cols = targets")
print(iou)
print("\nrow 0 vs col 0 = 1.000 (identical boxes); row 2 = 0 (no overlap at all)")

boxes1[:, None, :2] : (3, 1, 2)
boxes2[:, :2]       : (2, 2)
broadcast result    : (3, 2, 2)  <- every (pred, target) pair

IoU matrix (3, 2) -- rows = predictions, cols = targets
tensor([[1.000, 0.020],
        [0.143, 0.325],
        [0.000, 0.000]])

row 0 vs col 0 = 1.000 (identical boxes); row 2 = 0 (no overlap at all)


`.clamp(min=0)` is the other essential here: when two boxes do not overlap, `rb - lt` goes negative on both axes and their product would be a bogus **positive** area. Clamping to 0 first makes non-overlap mean zero intersection.

In [17]:
# What clamp is protecting you from
far_a = torch.tensor([[0., 0., 10., 10.]])
far_b = torch.tensor([[20., 20., 30., 30.]])

lt = torch.max(far_a[:, None, :2], far_b[:, :2])
rb = torch.min(far_a[:, None, 2:], far_b[:, 2:])
raw = rb - lt
print("rb - lt          =", raw.flatten().tolist(), " <- negative on both axes")
print("without clamp    =", (raw[..., 0] * raw[..., 1]).item(), " <- POSITIVE. Nonsense.")
print("with clamp(min=0)=", (raw.clamp(min=0)[..., 0] * raw.clamp(min=0)[..., 1]).item())

rb - lt          = [-10.0, -10.0]  <- negative on both axes
without clamp    = 100.0  <- POSITIVE. Nonsense.
with clamp(min=0)= 0.0


The same pairwise pattern shows up again in [`matcher.py:71`](../models/matcher.py#L71) as `torch.cdist(out_bbox, tgt_bbox, p=1)` — the all-pairs L1 distance, 100 predictions × N targets in a single call.

And once more in [`backbone.py:47-50`](../models/backbone.py#L47-L50), where `FrozenBatchNorm2d` reshapes a per-channel vector so it broadcasts over a whole feature map:

```python
w = self.weight.reshape(1, -1, 1, 1)     # (C,) -> (1, C, 1, 1)
return x * scale + bias                  # (B,C,H,W) * (1,C,1,1)
```

In [18]:
x = torch.randn(2, 8, 4, 4)      # (B, C, H, W)
per_channel = torch.arange(8, dtype=torch.float32)

print("x            ", tuple(x.shape))
print("per_channel  ", tuple(per_channel.shape))
try:
    x * per_channel
except RuntimeError as e:
    print("x * per_channel -> RuntimeError:", str(e).splitlines()[0][:62])

w = per_channel.reshape(1, -1, 1, 1)
print("\nper_channel.reshape(1,-1,1,1):", tuple(w.shape))
print("x * w        ->", tuple((x * w).shape), " each channel scaled by its own value")

x             (2, 8, 4, 4)
per_channel   (8,)
x * per_channel -> RuntimeError: The size of tensor a (4) must match the size of tensor b (8) a

per_channel.reshape(1,-1,1,1): (1, 8, 1, 1)
x * w        -> (2, 8, 4, 4)  each channel scaled by its own value


### `repeat` vs `expand`

[`transformer.py:52`](../models/transformer.py#L52) needs the same 100 object queries for every image in the batch:

```python
query_embed = query_embed.unsqueeze(1).repeat(1, bs, 1)   # (100,256) -> (100,B,256)
```

In [19]:
q = torch.randn(100, 256)                       # the learned queries, shared by all images
bs = 4

rep = q.unsqueeze(1).repeat(1, bs, 1)           # copies the data
exp = q.unsqueeze(1).expand(-1, bs, -1)         # a view: stride 0 on the batch axis

print("repeat:", tuple(rep.shape), "contiguous:", rep.is_contiguous(),
      "| floats in storage:", rep.untyped_storage().nbytes() // 4)
print("expand:", tuple(exp.shape), "contiguous:", exp.is_contiguous(),
      "| floats in storage:", exp.untyped_storage().nbytes() // 4)
print("expand strides:", exp.stride(), " <- the 0 is what makes it free")
print()
print("same values:", torch.equal(rep, exp))
print("expand allocates nothing; repeat allocates", bs, "x the memory.")
print("DETR uses repeat because attention layers write into the result,")
print("and writing into a stride-0 view would alias every batch element.")

repeat: (100, 4, 256) contiguous: True | floats in storage: 102400
expand: (100, 4, 256) contiguous: False | floats in storage: 25600
expand strides: (256, 0, 1)  <- the 0 is what makes it free

same values: True
expand allocates nothing; repeat allocates 4 x the memory.
DETR uses repeat because attention layers write into the result,
and writing into a stride-0 view would alias every batch element.


---

## 4. Indexing

### Negative indices and `...`

Two lines from [`models/detr.py`](../models/detr.py) that look alike and mean completely different things:

```python
out = {'pred_logits': outputs_class[-1], ...}        # detr.py:69  -- last DECODER LAYER
scores, labels = prob[..., :-1].max(-1)              # detr.py:275 -- drop the LAST CLASS
```

The first indexes axis 0 (the six decoder layers). The second uses `...` to skip every leading axis and slice the *class* axis. Same `-1`, two unrelated jobs.

In [20]:
outputs_class = torch.randn(6, 2, 100, 92)   # (dec_layers, B, queries, classes+1)

print("full tensor    ", tuple(outputs_class.shape))
print("[-1]           ", tuple(outputs_class[-1].shape),        " <- final decoder layer")
print("[:-1]          ", tuple(outputs_class[:-1].shape),       " <- the 5 aux-loss layers")
print("[..., :-1]     ", tuple(outputs_class[..., :-1].shape),  " <- drop the no-object class")
print("[-1][..., :-1] ", tuple(outputs_class[-1][..., :-1].shape))
print()
print("'...' means 'as many ':' as needed'. Here [..., :-1] == [:, :, :, :-1],")
print("but it keeps working when the number of leading axes changes -- which")
print("matters because PostProcess is called on (B,Q,C) while the loss sees")
print("the same slicing logic applied elsewhere.")

full tensor     (6, 2, 100, 92)
[-1]            (2, 100, 92)  <- final decoder layer
[:-1]           (5, 2, 100, 92)  <- the 5 aux-loss layers
[..., :-1]      (6, 2, 100, 91)  <- drop the no-object class
[-1][..., :-1]  (2, 100, 91)

'...' means 'as many ':' as needed'. Here [..., :-1] == [:, :, :, :-1],
but it keeps working when the number of leading axes changes -- which
matters because PostProcess is called on (B,Q,C) while the loss sees
the same slicing logic applied elsewhere.


### Boolean masks

`~mask` inverts a bool tensor. [`position_encoding.py:32-34`](../models/position_encoding.py#L32-L34) then uses `not_mask` as **numbers**, relying on `True == 1`:

In [21]:
mask = torch.zeros(1, 4, 6, dtype=torch.bool)
mask[:, :, 4:] = True                            # right two columns are padding

not_mask = ~mask
print("not_mask (1 = real pixel):")
print(not_mask[0].int())
print("\ncumsum along width -- the x coordinate, which STOPS at the padding:")
print(not_mask.cumsum(2)[0])
print("\nreal pixels:", not_mask.sum().item(), "/", not_mask.numel())
print("\nA bool tensor summed counts its Trues. No .float() needed.")

not_mask (1 = real pixel):
tensor([[1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0, 0]], dtype=torch.int32)

cumsum along width -- the x coordinate, which STOPS at the padding:
tensor([[1, 2, 3, 4, 4, 4],
        [1, 2, 3, 4, 4, 4],
        [1, 2, 3, 4, 4, 4],
        [1, 2, 3, 4, 4, 4]])

real pixels: 16 / 24

A bool tensor summed counts its Trues. No .float() needed.


### Advanced indexing: the one that stops people cold

This is the hardest indexing idiom in DETR, and it is the heart of the set loss. From [`detr.py:115-119`](../models/detr.py#L115-L119):

```python
idx = self._get_src_permutation_idx(indices)
target_classes = torch.full(src_logits.shape[:2], self.num_classes, dtype=torch.int64, ...)
target_classes[idx] = target_classes_o
```

**What is happening:** the Hungarian matcher returns, per image, which *query* was matched to which *target*. Those matched queries must receive the ground-truth class; every other query must receive "no object". `idx` is a **tuple of two index tensors**, and indexing with a tuple of tensors selects arbitrary scattered positions.

In [22]:
# What the matcher returns: per image, (query_indices, target_indices)
indices = [(torch.tensor([1, 3]), torch.tensor([0, 1])),   # image 0: query1<-tgt0, query3<-tgt1
           (torch.tensor([2]),    torch.tensor([0]))]      # image 1: query2<-tgt0
targets = [{"labels": torch.tensor([17, 18])},             # image 0: cat, dog
           {"labels": torch.tensor([3])}]                  # image 1: car

# --- _get_src_permutation_idx (detr.py:193-197) ---
batch_idx = torch.cat([torch.full_like(src, i) for i, (src, _) in enumerate(indices)])
src_idx   = torch.cat([src for (src, _) in indices])
idx = (batch_idx, src_idx)

print("batch_idx:", batch_idx.tolist(), " <- which image")
print("src_idx  :", src_idx.tolist(),   " <- which query")
print("read as pairs:", list(zip(batch_idx.tolist(), src_idx.tolist())))
print()
print("Note the flattening: image 0 contributes 2 entries, image 1 contributes 1.")
print("torch.cat handles the variable lengths; full_like supplies the image id.")

batch_idx: [0, 0, 1]  <- which image
src_idx  : [1, 3, 2]  <- which query
read as pairs: [(0, 1), (0, 3), (1, 2)]

Note the flattening: image 0 contributes 2 entries, image 1 contributes 1.
torch.cat handles the variable lengths; full_like supplies the image id.


In [23]:
B_, NUM_QUERIES, NUM_CLASSES = 2, 5, 91         # 5 queries instead of 100, to fit on screen

# every query starts as "no object" (= class index num_classes)
target_classes = torch.full((B_, NUM_QUERIES), NUM_CLASSES, dtype=torch.int64)
print("before -- everything is no-object (91):")
print(target_classes)

target_classes_o = torch.cat([t["labels"][J] for t, (_, J) in zip(targets, indices)])
print("\ntarget labels gathered in matcher order:", target_classes_o.tolist())

target_classes[idx] = target_classes_o          # <- the scatter
print("\nafter -- matched queries hold their class:")
print(target_classes)

before -- everything is no-object (91):
tensor([[91, 91, 91, 91, 91],
        [91, 91, 91, 91, 91]])

target labels gathered in matcher order: [17, 18, 3]

after -- matched queries hold their class:
tensor([[91, 17, 91, 18, 91],
        [91, 91,  3, 91, 91]])


Read the result: image 0's queries 1 and 3 hold 17 (cat) and 18 (dog); image 1's query 2 holds 3 (car). Everything else is 91 = ∅.

**The general rule:** `x[(i, j)]` where `i` and `j` are index tensors of the same length selects `x[i[0], j[0]], x[i[1], j[1]], ...`. It **reads** scattered elements, and — as here — it **writes** them too.

That `torch.full(shape, fill_value)` starting point is what makes the whole set loss work: **default everything to ∅, then overwrite the matched few.**

The same `idx` is reused to *read* in [`detr.py:150`](../models/detr.py#L150) — `src_boxes = outputs['pred_boxes'][idx]` pulls out exactly the matched predictions, in the same order as the concatenated targets, so the box loss compares the right pairs.

In [24]:
pred_boxes = torch.rand(B_, NUM_QUERIES, 4)      # (B, queries, 4)
src_boxes = pred_boxes[idx]                      # (3, 4) -- only the matched ones

print("pred_boxes", tuple(pred_boxes.shape), "-> pred_boxes[idx]", tuple(src_boxes.shape))
print()
print("Row order is guaranteed to line up with:")
print("  torch.cat([t['boxes'][i] for t, (_, i) in zip(targets, indices)])")
print("because both are built by walking `indices` in the same order.")
print("That alignment is the entire contract between matcher.py and the loss.")

pred_boxes (2, 5, 4) -> pred_boxes[idx] (3, 4)

Row order is guaranteed to line up with:
  torch.cat([t['boxes'][i] for t, (_, i) in zip(targets, indices)])
because both are built by walking `indices` in the same order.
That alignment is the entire contract between matcher.py and the loss.


---

## 5. Reductions

### `.max()` returns two things (and `.max(dim)` is not `.max()`)

The single most-copied line in DETR inference ([`detr.py:274-275`](../models/detr.py#L274-L275)):

```python
prob = F.softmax(out_logits, -1)
scores, labels = prob[..., :-1].max(-1)
```

In [25]:
logits = torch.randn(1, 5, 92)                  # 1 image, 5 queries, 92 classes
prob = F.softmax(logits, -1)

print("prob.max()   ->", prob.max(), "  <- NO dim: a single scalar")
out = prob[..., :-1].max(-1)
print("prob.max(-1) -> torch.return_types." + type(out).__name__, "-- a PAIR of tensors:")
print("   .values ", tuple(out.values.shape))
print("   .indices", tuple(out.indices.shape))
print()

scores, labels = prob[..., :-1].max(-1)
print("scores", tuple(scores.shape), "= the confidence")
print("labels", tuple(labels.shape), "= the argmax index = the class id")
print()
for q, (s, l) in enumerate(zip(scores[0], labels[0])):
    print(f"  query {q}: class {l.item():>2} ({COCO_CLASSES[l]:<14}) score {s.item():.3f}")
print()
print("(Untrained random logits, so the scores are all ~1/92. In a real model")
print(" the kept detections sit above 0.7 and the rest collapse onto no-object.)")

prob.max()   -> tensor(0.095)   <- NO dim: a single scalar
prob.max(-1) -> torch.return_types.max -- a PAIR of tensors:
   .values  (1, 5)
   .indices (1, 5)

scores (1, 5) = the confidence
labels (1, 5) = the argmax index = the class id

  query 0: class 87 (scissors      ) score 0.095
  query 1: class 23 (bear          ) score 0.076
  query 2: class 29 (N/A           ) score 0.063
  query 3: class 35 (skis          ) score 0.047
  query 4: class 61 (cake          ) score 0.095

(Untrained random logits, so the scores are all ~1/92. In a real model
 the kept detections sit above 0.7 and the rest collapse onto no-object.)


`.max(dim)` returns a named tuple `(values, indices)`; `.max()` with no dim returns a bare scalar. Unpacking the two is why one line yields both a label *and* a confidence.

If you only want the index, `.argmax(-1)` is the same thing with the values discarded — [`detr.py:138`](../models/detr.py#L138) counts non-∅ predictions that way:

```python
card_pred = (pred_logits.argmax(-1) != pred_logits.shape[-1] - 1).sum(1)
```

In [26]:
pred_logits = torch.randn(2, 100, 92)
card_pred = (pred_logits.argmax(-1) != pred_logits.shape[-1] - 1).sum(1)

print("argmax(-1)      ", tuple(pred_logits.argmax(-1).shape), " one class id per query")
print("!= 91           ", tuple((pred_logits.argmax(-1) != 91).shape), " a bool tensor")
print("        .sum(1) ", tuple(card_pred.shape), " -> per-image count:", card_pred.tolist())
print()
print("Three ops, one metric. `pred_logits.shape[-1] - 1` rather than a literal 91")
print("is what lets the same line work for a 21-class VOC model.")

argmax(-1)       (2, 100)  one class id per query
!= 91            (2, 100)  a bool tensor
        .sum(1)  (2,)  -> per-image count: [98, 98]

Three ops, one metric. `pred_logits.shape[-1] - 1` rather than a literal 91
is what lets the same line work for a 21-class VOC model.


### `dim` and `keepdim`

`dim` names the axis that **disappears**. `keepdim=True` leaves it behind as size 1, which is exactly what you need when the result must broadcast back against the original.

In [27]:
x = torch.randn(2, 3, 4)
print("x                      ", tuple(x.shape))
print("x.sum(0)               ", tuple(x.sum(0).shape),  " axis 0 gone")
print("x.sum(-1)              ", tuple(x.sum(-1).shape), " last axis gone")
print("x.sum(-1, keepdim=True)", tuple(x.sum(-1, keepdim=True).shape), " kept as 1")
print()
print("normalizing needs keepdim (or it won't line up):")
print("  x / x.sum(-1, keepdim=True) ->", tuple((x / x.sum(-1, keepdim=True)).shape))
try:
    (x / x.sum(-1)).shape
except RuntimeError as e:
    print("  x / x.sum(-1)               -> RuntimeError:", str(e).splitlines()[0][:52])

x                       (2, 3, 4)
x.sum(0)                (3, 4)  axis 0 gone
x.sum(-1)               (2, 3)  last axis gone
x.sum(-1, keepdim=True) (2, 3, 1)  kept as 1

normalizing needs keepdim (or it won't line up):
  x / x.sum(-1, keepdim=True) -> (2, 3, 4)
  x / x.sum(-1)               -> RuntimeError: The size of tensor a (4) must match the size of tens


### `cumsum`, and the position encoding's normalization

[`position_encoding.py:33-38`](../models/position_encoding.py#L33-L38) combines everything above. `cumsum` builds coordinates that respect padding, then a slice that keeps its axis normalizes each image by *its own* extent:

```python
y_embed = not_mask.cumsum(1, dtype=torch.float32)
x_embed = not_mask.cumsum(2, dtype=torch.float32)
y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale
```

Note `x_embed[:, :, -1:]` — the slice `-1:` keeps the axis (size 1), where `-1` would drop it. That is `keepdim` by another name.

In [28]:
mask = torch.zeros(1, 3, 6, dtype=torch.bool)
mask[:, :, 4:] = True                                   # last two columns padded

not_mask = ~mask
x_embed = not_mask.cumsum(2, dtype=torch.float32)
print("x_embed (raw cumsum) -- note it PLATEAUS over the padding:")
print(x_embed[0])

print("\nthe divisor, x_embed[:, :, -1:]:", tuple(x_embed[:, :, -1:].shape), " <- axis kept")
print("      compare x_embed[:, :,  -1]:", tuple(x_embed[:, :, -1].shape),  " <- axis dropped")

x_norm = x_embed / (x_embed[:, :, -1:] + 1e-6) * (2 * math.pi)
print("\nx_embed normalized to (0, 2*pi]:")
print(x_norm[0])
print("\nEvery image ends at 2*pi regardless of its width. That is what makes")
print("DETR's position encoding resolution-independent.")

x_embed (raw cumsum) -- note it PLATEAUS over the padding:
tensor([[1., 2., 3., 4., 4., 4.],
        [1., 2., 3., 4., 4., 4.],
        [1., 2., 3., 4., 4., 4.]])

the divisor, x_embed[:, :, -1:]: (1, 3, 1)  <- axis kept
      compare x_embed[:, :,  -1]: (1, 3)  <- axis dropped

x_embed normalized to (0, 2*pi]:
tensor([[1.571, 3.142, 4.712, 6.283, 6.283, 6.283],
        [1.571, 3.142, 4.712, 6.283, 6.283, 6.283],
        [1.571, 3.142, 4.712, 6.283, 6.283, 6.283]])

Every image ends at 2*pi regardless of its width. That is what makes
DETR's position encoding resolution-independent.


### `topk` and the accuracy metric

[`util/misc.py:433-447`](../util/misc.py#L433-L447) computes top-k accuracy with `topk`, `t()`, `eq()` and `expand_as` — a compact idiom worth being able to read:

```python
_, pred = output.topk(maxk, 1, True, True)      # (N, k) indices of the k best classes
pred = pred.t()                                 # (k, N)
correct = pred.eq(target.view(1, -1).expand_as(pred))
```

In [29]:
output = torch.randn(4, 10)          # 4 samples, 10 classes
target = torch.tensor([1, 2, 3, 4])

_, pred = output.topk(3, 1, True, True)   # k=3, dim=1, largest=True, sorted=True
print("topk(3) indices per sample:\n", pred)
pred_t = pred.t()
print("\nafter .t():", tuple(pred_t.shape), " (k, N) -- one row per rank")

correct = pred_t.eq(target.view(1, -1).expand_as(pred_t))
print("\ntarget broadcast to (k, N):\n", target.view(1, -1).expand_as(pred_t))
print("\ncorrect:\n", correct)
print("\ntop-1 accuracy:", correct[:1].reshape(-1).float().sum(0).mul_(100/4).item(), "%")
print("top-3 accuracy:", correct[:3].reshape(-1).float().sum(0).mul_(100/4).item(), "%")

topk(3) indices per sample:
 tensor([[2, 3, 4],
        [6, 7, 5],
        [4, 3, 0],
        [8, 2, 5]])

after .t(): (3, 4)  (k, N) -- one row per rank

target broadcast to (k, N):
 tensor([[1, 2, 3, 4],
        [1, 2, 3, 4],
        [1, 2, 3, 4]])

correct:
 tensor([[False, False, False, False],
        [False, False,  True, False],
        [False, False, False, False]])

top-1 accuracy: 0.0 %
top-3 accuracy: 25.0 %


---

## 6. Matmul, and the shape of attention

`@` is matrix multiply. On tensors with more than two axes it **batches over the leading axes** and multiplies the last two. That single rule is what lets attention process a whole batch of heads at once with no loop.

In [30]:
print("2-D:        (4,8) @ (8,3)          ->", tuple((torch.randn(4,8) @ torch.randn(8,3)).shape))
print("batched:  (2,4,8) @ (2,8,3)        ->", tuple((torch.randn(2,4,8) @ torch.randn(2,8,3)).shape))
print("broadcast:(2,4,8) @   (8,3)        ->", tuple((torch.randn(2,4,8) @ torch.randn(8,3)).shape))
print("2 batch axes: (2,3,4,8) @ (2,3,8,5)->",
      tuple((torch.randn(2,3,4,8) @ torch.randn(2,3,8,5)).shape))
print()
print("The last two axes are the matrix. Everything to the left is batch.")

2-D:        (4,8) @ (8,3)          -> (4, 3)
batched:  (2,4,8) @ (2,8,3)        -> (2, 4, 3)
broadcast:(2,4,8) @   (8,3)        -> (2, 4, 3)
2 batch axes: (2,3,4,8) @ (2,3,8,5)-> (2, 3, 4, 5)

The last two axes are the matrix. Everything to the left is batch.


### `transpose(-2, -1)` — why not `.T`

Attention scores are `Q @ K^T`. You cannot write `k.T` on a 3-D tensor (it is deprecated and means something else), so the idiom is **`k.transpose(-2, -1)`**: swap the last two axes, whatever the rank.

In [31]:
L, D = 4, 8
q = torch.randn(L, D)
k = torch.randn(L, D)

scores = q @ k.transpose(-2, -1) / math.sqrt(D)
print("q", tuple(q.shape), "@ k^T", tuple(k.transpose(-2,-1).shape), "->", tuple(scores.shape))
print("attention weights = softmax over the LAST axis (the keys):")
attn = scores.softmax(-1)
print(attn)
print("\nrows sum to 1:", [round(v, 4) for v in attn.sum(-1).tolist()])
print()
# and it works unchanged with batch + head axes in front
qb = torch.randn(2, 8, L, D)                      # (batch, heads, seq, head_dim)
kb = torch.randn(2, 8, L, D)
print("batched+heads:", tuple((qb @ kb.transpose(-2, -1)).shape), " (B, heads, L, L)")

q (4, 8) @ k^T (8, 4) -> (4, 4)
attention weights = softmax over the LAST axis (the keys):
tensor([[0.465, 0.318, 0.176, 0.041],
        [0.180, 0.449, 0.060, 0.310],
        [0.242, 0.012, 0.732, 0.014],
        [0.128, 0.191, 0.105, 0.576]])

rows sum to 1: [1.0, 1.0, 1.0, 1.0]

batched+heads: (2, 8, 4, 4)  (B, heads, L, L)


**The softmax axis is the thing to get right.** `softmax(-1)` normalizes over *keys* — each query's weights sum to 1. `softmax(-2)` would normalize over queries, which is meaningless. Notebook [`01`](01_attention_from_scratch.ipynb) builds this out in full; the point here is just that `-1` is a decision, not a default.

Two other softmaxes in DETR, both over the class axis and both `-1` for the same structural reason:

```python
prob = F.softmax(out_logits, -1)                          # detr.py:274
out_prob = outputs["pred_logits"].flatten(0, 1).softmax(-1)   # matcher.py:58
```

Note `flatten(0, 1)` there — the **two-argument** form of flatten, merging only axes 0 and 1: `(B, 100, 92) -> (B*100, 92)`. The matcher works on one flat pile of predictions and splits it back per image afterwards.

In [32]:
pred = torch.randn(2, 100, 92)
print("flatten(0, 1):", tuple(pred.shape), "->", tuple(pred.flatten(0, 1).shape))
print("flatten(2)   :", tuple(pred.shape), "->", tuple(pred.flatten(2).shape), " (no-op here)")
print()
print("matcher.py:77  C.view(bs, num_queries, -1)   splits it back:")
C = torch.randn(200, 3)
print("  ", tuple(C.shape), "->", tuple(C.view(2, 100, -1).shape))

flatten(0, 1): (2, 100, 92) -> (200, 92)
flatten(2)   : (2, 100, 92) -> (2, 100, 92)  (no-op here)

matcher.py:77  C.view(bs, num_queries, -1)   splits it back:
   (200, 3) -> (2, 100, 3)


---

## 7. The four layers that carry weights

DETR's whole parameter budget lives in four kinds of layer. Knowing what each one does to shapes is most of reading the model.

In [33]:
from models.detr import MLP

hidden_dim, num_classes, num_queries = 256, 91, 100

class_embed = nn.Linear(hidden_dim, num_classes + 1)          # detr.py:37
bbox_embed  = MLP(hidden_dim, hidden_dim, 4, 3)               # detr.py:38
query_embed = nn.Embedding(num_queries, hidden_dim)           # detr.py:39
input_proj  = nn.Conv2d(2048, hidden_dim, kernel_size=1)      # detr.py:40

for name, mod in [("class_embed", class_embed), ("bbox_embed", bbox_embed),
                  ("query_embed", query_embed), ("input_proj", input_proj)]:
    n = sum(p.numel() for p in mod.parameters())
    print(f"{name:<12} {type(mod).__name__:<10} {n:>10,} params")

class_embed  Linear         23,644 params
bbox_embed   MLP           132,612 params
query_embed  Embedding      25,600 params
input_proj   Conv2d        524,544 params


### `nn.Linear` applies to the **last** axis only

This is the property that makes `class_embed` work on `(6, B, 100, 256)` without any reshaping: a `Linear` treats every leading axis as batch.

In [34]:
hs = torch.randn(6, 2, 100, 256)          # (dec_layers, B, queries, hidden)

print("hs                     ", tuple(hs.shape))
print("class_embed(hs)        ", tuple(class_embed(hs).shape), " last axis 256 -> 92")
print("bbox_embed(hs).sigmoid()", tuple(bbox_embed(hs).sigmoid().shape), " last axis 256 -> 4")
print()
print("No flattening, no loop over layers or queries. Linear maps the last")
print("axis and broadcasts over all", hs.ndim - 1, "leading axes.")
print()
print("That is exactly detr.py:67-68:")
print("  outputs_class = self.class_embed(hs)")
print("  outputs_coord = self.bbox_embed(hs).sigmoid()")

hs                      (6, 2, 100, 256)
class_embed(hs)         (6, 2, 100, 92)  last axis 256 -> 92
bbox_embed(hs).sigmoid() (6, 2, 100, 4)  last axis 256 -> 4

No flattening, no loop over layers or queries. Linear maps the last
axis and broadcasts over all 3 leading axes.

That is exactly detr.py:67-68:
  outputs_class = self.class_embed(hs)
  outputs_coord = self.bbox_embed(hs).sigmoid()


`.sigmoid()` on the box head is not decoration: it forces all four coordinates into `(0, 1)`, which is precisely the normalized `cxcywh` format the targets use. The classification head deliberately has **no** activation — see §11.

### `nn.Conv2d(..., kernel_size=1)` is a per-pixel `Linear`

`input_proj` squeezes ResNet's 2048 channels to 256. A 1×1 convolution touches no neighbours at all — it is a `Linear` applied independently at every spatial position, just expressed in channels-first layout.

In [35]:
conv = nn.Conv2d(8, 4, kernel_size=1)
x = torch.randn(1, 8, 3, 3)

# the equivalent Linear, sharing the same weights
lin = nn.Linear(8, 4)
with torch.no_grad():
    lin.weight.copy_(conv.weight[:, :, 0, 0])
    lin.bias.copy_(conv.bias)

via_conv = conv(x)                                      # (1, 4, 3, 3)
via_lin  = lin(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)

print("conv weight shape:", tuple(conv.weight.shape), " <- the 1x1 spatial extent")
print("lin  weight shape:", tuple(lin.weight.shape))
print("identical outputs:", torch.allclose(via_conv, via_lin, atol=1e-6))
print()
print("So input_proj is 'project each of the 850 positions from 2048-d to 256-d'.")

conv weight shape: (4, 8, 1, 1)  <- the 1x1 spatial extent
lin  weight shape: (4, 8)
identical outputs: True

So input_proj is 'project each of the 850 positions from 2048-d to 256-d'.


### `nn.Embedding` is a lookup table — and DETR uses it sideways

Normally you call an `Embedding` with integer ids. DETR never does. It reads `.weight` **directly** ([`detr.py:65`](../models/detr.py#L65)):

```python
hs = self.transformer(self.input_proj(src), mask, self.query_embed.weight, pos[-1])[0]
```

The object queries are not looked up by anything — all 100 rows are used, every forward pass. `nn.Embedding` is simply a convenient container for a learned `(100, 256)` parameter matrix.

In [36]:
emb = nn.Embedding(100, 256)

print("emb.weight          :", tuple(emb.weight.shape), "| nn.Parameter:",
      isinstance(emb.weight, nn.Parameter))
print("emb(tensor([0, 5])) :", tuple(emb(torch.tensor([0, 5])).shape), " the usual lookup use")
print("equals emb.weight[[0,5]]:", torch.equal(emb(torch.tensor([0, 5])), emb.weight[[0, 5]]))
print()
print("DETR just takes the whole table:", tuple(emb.weight.shape))
print("-> 100 learned 256-d vectors, identical for every image.")
print("   They are positional embeddings for the decoder. See notebook 03.")

emb.weight          : (100, 256) | nn.Parameter: True
emb(tensor([0, 5])) : (2, 256)  the usual lookup use
equals emb.weight[[0,5]]: True

DETR just takes the whole table: (100, 256)
-> 100 learned 256-d vectors, identical for every image.
   They are positional embeddings for the decoder. See notebook 03.


### `nn.MultiheadAttention` and the sequence-first convention

This DETR predates `batch_first=True`, so **every** transformer tensor is `(sequence, batch, embed)` — batch is in the *middle*. That is why §2's `permute(2, 0, 1)` puts `HW` first.

In [37]:
mha = nn.MultiheadAttention(embed_dim=8, num_heads=2)     # batch_first defaults to False

q = torch.randn(3, 2, 8)         # (tgt_len=3, batch=2, embed=8)
kv = torch.randn(5, 2, 8)        # (src_len=5, batch=2, embed=8)

out, weights = mha(q, kv, kv)
print("query  ", tuple(q.shape),  " (L, B, E)")
print("key/val", tuple(kv.shape), " (S, B, E)")
print("output ", tuple(out.shape), " (L, B, E)  -- one vector per QUERY")
print("weights", tuple(weights.shape), " (B, L, S)  -- averaged over heads by default")
print()
print("Cross-attention in a nutshell: L queries, S keys, output length follows")
print("the QUERIES. That is how 850 image tokens become 100 detections.")

query   (3, 2, 8)  (L, B, E)
key/val (5, 2, 8)  (S, B, E)
output  (3, 2, 8)  (L, B, E)  -- one vector per QUERY
weights (2, 3, 5)  (B, L, S)  -- averaged over heads by default

Cross-attention in a nutshell: L queries, S keys, output length follows
the QUERIES. That is how 850 image tokens become 100 detections.


In [38]:
# key_padding_mask: True means "ignore this position"
kpm = torch.zeros(2, 5, dtype=torch.bool)
kpm[1, 3:] = True                       # batch item 1 has only 3 real tokens

out, weights = mha(q, kv, kv, key_padding_mask=kpm)
print("attention weights for batch item 1 (padded positions 3,4):")
print(weights[1])
print("\nlast two columns are exactly 0 ->", torch.allclose(weights[1][:, 3:],
                                                            torch.zeros(3, 2), atol=1e-7))
print()
print("This is the padding mask from util/misc.py, threaded all the way through")
print("transformer.py:56 as src_key_padding_mask. Same bool tensor, same polarity.")

attention weights for batch item 1 (padded positions 3,4):
tensor([[0.171, 0.225, 0.604, 0.000, 0.000],
        [0.280, 0.316, 0.404, 0.000, 0.000],
        [0.316, 0.341, 0.344, 0.000, 0.000]], grad_fn=<SelectBackward0>)

last two columns are exactly 0 -> True

This is the padding mask from util/misc.py, threaded all the way through
transformer.py:56 as src_key_padding_mask. Same bool tensor, same polarity.


---

## 8. `nn.Module`: the container protocol

Three rules cover most of what you need:

1. Call `super().__init__()` **first**, before assigning anything.
2. Assign submodules and `nn.Parameter`s as **attributes** — that is how they get registered.
3. Define `forward`, but **call the module**: `model(x)`, never `model.forward(x)` (hooks would be skipped).

[`detr.py:33-42`](../models/detr.py#L33-L42) is a textbook example — attribute assignments become registered children:

In [39]:
from models.transformer import build_transformer

transformer = build_transformer(detr_args())        # no download: pure torch
head = MLP(256, 256, 4, 3)                          # the bbox head from detr.py:38

print("MLP children (an nn.ModuleList of 3 Linears):")
for name, child in head.named_children():
    print("  ", name, "->", child)
print()
total = sum(p.numel() for p in transformer.parameters())
train = sum(p.numel() for p in transformer.parameters() if p.requires_grad)
print(f"transformer parameters: {total:,} total, {train:,} trainable")
print()
print("Top-level children of the transformer:")
for name, child in transformer.named_children():
    n = sum(p.numel() for p in child.parameters())
    print(f"   {name:<10} {type(child).__name__:<20} {n:>12,} params")

MLP children (an nn.ModuleList of 3 Linears):
   layers -> ModuleList(
  (0-1): 2 x Linear(in_features=256, out_features=256, bias=True)
  (2): Linear(in_features=256, out_features=4, bias=True)
)

transformer parameters: 17,363,456 total, 17,363,456 trainable

Top-level children of the transformer:
   encoder    TransformerEncoder      7,890,432 params
   decoder    TransformerDecoder      9,473,024 params


### Why `nn.ModuleList` and not a plain list

This is the most common structural bug in PyTorch. A Python list of modules is **invisible** to the parent: its parameters are not in `.parameters()`, not in `.state_dict()`, and `.to(device)` will not move them. The model trains, and silently never updates those layers.

[`transformer.py:272-273`](../models/transformer.py#L272-L273) gets it right:

```python
def _get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for i in range(N)])
```

In [40]:
class Wrong(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(4, 4) for _ in range(3)]        # plain list -- invisible

class Right(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList(nn.Linear(4, 4) for _ in range(3))

print("plain list  ->", sum(p.numel() for p in Wrong().parameters()), "parameters  <- BUG")
print("ModuleList  ->", sum(p.numel() for p in Right().parameters()), "parameters")
print()
print("state_dict keys, plain list:", list(Wrong().state_dict().keys()))
print("state_dict keys, ModuleList:", list(Right().state_dict().keys())[:2], "...")
print()
print("Both forward() identically. Only one of them trains -- or saves.")

plain list  -> 0 parameters  <- BUG
ModuleList  -> 60 parameters

state_dict keys, plain list: []
state_dict keys, ModuleList: ['layers.0.weight', 'layers.0.bias'] ...

Both forward() identically. Only one of them trains -- or saves.


`copy.deepcopy` in `_get_clones` matters too: the six encoder layers must be **independent** copies. Appending the *same* module six times would give you one layer applied six times with shared weights — a completely different (and much weaker) model.

In [41]:
import copy

layer = nn.Linear(4, 4)
shared = nn.ModuleList([layer for _ in range(3)])                    # WRONG
cloned = nn.ModuleList([copy.deepcopy(layer) for _ in range(3)])     # right

print("shared: ", sum(p.numel() for p in shared.parameters()), "params",
      "| distinct objects:", len({id(m) for m in shared}))
print("cloned: ", sum(p.numel() for p in cloned.parameters()), "params",
      "| distinct objects:", len({id(m) for m in cloned}))
print()
print("PyTorch de-duplicates shared parameters, so the first line's count is")
print("1/3 of the second. A 6-layer encoder built that way has the capacity of 1.")

shared:  20 params | distinct objects: 1
cloned:  60 params | distinct objects: 3

PyTorch de-duplicates shared parameters, so the first line's count is
1/3 of the second. A 6-layer encoder built that way has the capacity of 1.


`nn.Sequential` is a `ModuleList` that also defines `forward` (chaining its children). DETR subclasses it for `Joiner` ([`backbone.py:95-109`](../models/backbone.py#L95-L109)) purely to get registration plus `self[0]` / `self[1]` access, then **overrides** `forward` — because the two children are not chained. The backbone's output feeds the position encoder, but *both* results are returned:

```python
class Joiner(nn.Sequential):
    def forward(self, tensor_list):
        xs = self[0](tensor_list)          # the ResNet
        ...
        pos.append(self[1](x))             # the position encoding
        return out, pos
```

That is exactly why `features, pos = self.backbone(samples)` at [`detr.py:61`](../models/detr.py#L61) unpacks **two** values from what looks like a single module.

In [42]:
class Pair(nn.Sequential):                     # the Joiner pattern, minimized
    def __init__(self, a, b):
        super().__init__(a, b)
    def forward(self, x):
        y = self[0](x)
        return y, self[1](y)                   # both, not chained

p = Pair(nn.Linear(4, 6), nn.Linear(6, 2))
a, b = p(torch.randn(1, 4))
print("returns two things:", tuple(a.shape), tuple(b.shape))
print("children still registered:", [n for n, _ in p.named_children()])
print("params:", sum(q.numel() for q in p.parameters()))

returns two things: (1, 6) (1, 2)
children still registered: ['0', '1']
params: 44


### Parameters vs buffers vs plain attributes

| | trained? | in `state_dict()`? | moved by `.to()`? |
|---|---|---|---|
| `nn.Parameter` | yes | yes | yes |
| `register_buffer` | **no** | yes | yes |
| plain attribute | no | **no** | **no** |

[`backbone.py:29-32`](../models/backbone.py#L29-L32) uses buffers for frozen BatchNorm statistics — they must be saved and loaded with the checkpoint, but never receive gradients. [`detr.py:104-106`](../models/detr.py#L104-L106) uses one for the class-weight vector that down-weights ∅.

The third row is the trap: a plain `self.something = torch.ones(5)` stays on the CPU when you call `model.cuda()`, and produces a device-mismatch error deep in `forward`.

In [43]:
from models.backbone import FrozenBatchNorm2d

bn = FrozenBatchNorm2d(8)
print("parameters:", list(dict(bn.named_parameters()).keys()), " <- empty!")
print("buffers   :", list(dict(bn.named_buffers()).keys()))
print("state_dict:", list(bn.state_dict().keys()), " <- saved anyway")
print()

class Trap(nn.Module):
    def __init__(self):
        super().__init__()
        self.registered = nn.Parameter(torch.ones(3))
        self.buffered   = torch.ones(3)
        self.register_buffer("proper_buffer", torch.ones(3))
        self.plain      = torch.ones(3)          # <- invisible

t = Trap()
print("Trap.state_dict():", list(t.state_dict().keys()))
print("'plain' is missing -- it will not move with .to(device), and will not save.")

parameters: []  <- empty!
buffers   : ['weight', 'bias', 'running_mean', 'running_var']
state_dict: ['weight', 'bias', 'running_mean', 'running_var']  <- saved anyway

Trap.state_dict(): ['registered', 'proper_buffer']
'plain' is missing -- it will not move with .to(device), and will not save.


In [44]:
# SetCriterion's empty_weight (detr.py:104-106)
num_classes_, eos_coef = 91, 0.1
empty_weight = torch.ones(num_classes_ + 1)
empty_weight[-1] = eos_coef

print("empty_weight[:3] =", empty_weight[:3].tolist(), "... [-1] =", round(empty_weight[-1].item(), 2))
print()
print("-> the no-object class contributes 10x less to the loss, because ~90 of")
print("   100 queries are no-object and would otherwise dominate the gradient.")
print("   It is a buffer, not a parameter: fixed by --eos_coef, never learned.")

empty_weight[:3] = [1.0, 1.0, 1.0] ... [-1] = 0.1

-> the no-object class contributes 10x less to the loss, because ~90 of
   100 queries are no-object and would otherwise dominate the gradient.
   It is a buffer, not a parameter: fixed by --eos_coef, never learned.


### `train()` / `eval()`

`model.eval()` flips a flag (`self.training`) that changes what Dropout and BatchNorm do. It does **not** disable gradients — that is `torch.no_grad()`'s job, and you almost always want both.

DETR uses `FrozenBatchNorm2d` precisely so BatchNorm behaves identically in both modes (detection batches are far too small for reliable batch statistics), but the transformer's `nn.Dropout(0.1)` does still care.

In [45]:
drop = nn.Dropout(0.5)
x = torch.ones(8)

drop.train(); print("train mode:", drop(x).tolist())
drop.eval();  print("eval  mode:", drop(x).tolist(), " <- identity")
print()
print("module.training is the flag each module reads:", drop.training)
print(".eval() is recursive -- it sets the flag on every descendant.")
print()
print("Note train mode scales the survivors by 1/(1-p) = 2.0, so the expected")
print("value matches eval mode. That is 'inverted dropout'.")

train mode: [2.0, 0.0, 0.0, 0.0, 0.0, 2.0, 0.0, 2.0]
eval  mode: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]  <- identity

module.training is the flag each module reads: False
.eval() is recursive -- it sets the flag on every descendant.

Note train mode scales the survivors by 1/(1-p) = 2.0, so the expected
value matches eval mode. That is 'inverted dropout'.


---

## 9. Autograd

Every tensor with `requires_grad=True` records the operations applied to it. `.backward()` walks that graph backwards and **accumulates** `.grad` on the leaves.

In [46]:
w = torch.randn(3, requires_grad=True)
x = torch.randn(3)

y = (w * x).sum()
print("y:", round(y.item(), 4), " grad_fn:", y.grad_fn)

y.backward()
print("w.grad:", w.grad, "  (== x:", torch.allclose(w.grad, x), ")")

y2 = (w * x).sum(); y2.backward()
print("after a SECOND backward, w.grad:", w.grad, " <- ACCUMULATED, not replaced")
print()
print("That accumulation is why every training loop calls optimizer.zero_grad().")
print("It is a feature (it enables gradient accumulation over micro-batches),")
print("but it is a silent 2x on your gradients if you forget.")

y: 0.8516  grad_fn: <SumBackward0 object at 0x11cdfbac0>


w.grad: tensor([ 0.492, -1.232,  0.249])   (== x: True )
after a SECOND backward, w.grad: tensor([ 0.985, -2.464,  0.498])  <- ACCUMULATED, not replaced

That accumulation is why every training loop calls optimizer.zero_grad().
It is a feature (it enables gradient accumulation over micro-batches),
but it is a silent 2x on your gradients if you forget.


### Freezing

[`backbone.py:62-64`](../models/backbone.py#L62-L64) freezes ResNet's early layers — `conv1` and `layer1` are generic edge detectors that need no retraining, and freezing them saves a lot of activation memory:

```python
for name, parameter in backbone.named_parameters():
    if not train_backbone or 'layer2' not in name and 'layer3' not in name and 'layer4' not in name:
        parameter.requires_grad_(False)
```

In [47]:
net = nn.Sequential(nn.Linear(4, 4), nn.Linear(4, 4))

for name, p in net.named_parameters():
    if name.startswith('0.'):
        p.requires_grad_(False)                     # trailing _ : in place

for name, p in net.named_parameters():
    print(f"  {name:<12} requires_grad={p.requires_grad}")

trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f"\ntrainable: {trainable} / {sum(p.numel() for p in net.parameters())}")

  0.weight     requires_grad=False
  0.bias       requires_grad=False
  1.weight     requires_grad=True
  1.bias       requires_grad=True

trainable: 20 / 40


Freezing is not all-or-nothing in DETR. `main.py` builds **two parameter groups** so the pretrained backbone learns 10× slower than the freshly initialized transformer:

```python
param_dicts = [
    {"params": [p for n, p in model.named_parameters() if "backbone" not in n and p.requires_grad]},
    {"params": [p for n, p in model.named_parameters() if "backbone" in n and p.requires_grad],
     "lr": args.lr_backbone},                       # 1e-5 vs the default 1e-4
]
optimizer = torch.optim.AdamW(param_dicts, lr=args.lr, weight_decay=args.weight_decay)
```

In [48]:
model = nn.ModuleDict({"backbone": nn.Linear(4, 4), "transformer": nn.Linear(4, 4)})

param_dicts = [
    {"params": [p for n, p in model.named_parameters() if "backbone" not in n]},
    {"params": [p for n, p in model.named_parameters() if "backbone" in n], "lr": 1e-5},
]
opt = torch.optim.AdamW(param_dicts, lr=1e-4)

for i, g in enumerate(opt.param_groups):
    print(f"group {i}: lr={g['lr']:<8} {sum(p.numel() for p in g['params'])} params")
print()
print("Note `and p.requires_grad` in the real code: frozen params must be")
print("excluded, or AdamW will keep weight-decaying them toward zero even")
print("though they receive no gradient.")

group 0: lr=0.0001   20 params
group 1: lr=1e-05    20 params

Note `and p.requires_grad` in the real code: frozen params must be
excluded, or AdamW will keep weight-decaying them toward zero even
though they receive no gradient.


### `no_grad` and `detach`

`torch.no_grad()` stops the graph being built at all — less memory, faster. It is both a context manager and a decorator; [`detr.py:260`](../models/detr.py#L260) and [`detr.py:129`](../models/detr.py#L129) use the decorator form:

```python
@torch.no_grad()
def forward(self, outputs, target_sizes):    # PostProcess
```

`.detach()` cuts one tensor out of the graph while leaving the rest intact.

In [49]:
w = torch.randn(3, requires_grad=True)

y = (w * 2).sum()
print("normal         : requires_grad =", y.requires_grad, " grad_fn =", y.grad_fn is not None)

with torch.no_grad():
    y = (w * 2).sum()
    print("in no_grad()   : requires_grad =", y.requires_grad, " grad_fn =", y.grad_fn is not None)

y = (w.detach() * 2).sum()
print("after .detach(): requires_grad =", y.requires_grad)
print()
print("Rule: inference and metrics -> no_grad. Logging a loss value -> .item().")

normal         : requires_grad = True  grad_fn = True
in no_grad()   : requires_grad = False  grad_fn = False
after .detach(): requires_grad = False

Rule: inference and metrics -> no_grad. Logging a loss value -> .item().


The matcher shows why this matters structurally. [`matcher.py:34`](../models/matcher.py#L34) wraps the **entire** Hungarian assignment in `@torch.no_grad()`. The matching decides *which* prediction is compared to *which* target — a discrete combinatorial choice, not something you differentiate through. Gradients flow only through the loss computed **after** the assignment.

`loss_cardinality` ([`detr.py:129`](../models/detr.py#L129)) is decorated for a different reason: it is a logging metric, not a loss. The docstring says so outright — *"It doesn't propagate gradients."*

In [50]:
# Gradient clipping, the last autograd ingredient DETR uses (main.py)
net = nn.Linear(4, 4)
loss = net(torch.randn(8, 4)).pow(2).mean()
loss.backward()

before = torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=0.1)
after = torch.sqrt(sum(p.grad.pow(2).sum() for p in net.parameters()))
print(f"grad norm before clipping: {before:.4f}")
print(f"grad norm after  clipping: {after:.4f}   (capped at 0.1)")
print()
print("DETR trains with --clip_max_norm 0.1. Transformers are prone to")
print("occasional huge gradients early in training; clipping keeps one bad")
print("batch from destroying the run.")

grad norm before clipping: 0.8010
grad norm after  clipping: 0.1000   (capped at 0.1)

DETR trains with --clip_max_norm 0.1. Transformers are prone to
occasional huge gradients early in training; clipping keeps one bad
batch from destroying the run.


---

## 10. `nn.Something` vs `F.something`

The same operation often appears both ways:

- **`nn.X` is a class holding state** — weights (`nn.Linear`), or a mode flag (`nn.Dropout` must know train vs eval). Create it in `__init__`.
- **`F.x` is a plain function** — no state, no registration. Call it in `forward`.

[`models/transformer.py`](../models/transformer.py) uses both deliberately: `nn.Linear`, `nn.Dropout`, `nn.LayerNorm`, `nn.MultiheadAttention` as attributes; `F.relu` inline.

In [51]:
lin = nn.Linear(4, 4)
print("nn.Linear state :", [n for n, _ in lin.named_parameters()])
print("F.relu state    : none -- it is just max(x, 0)")
print()

x = torch.randn(4)
print("F.relu(x)    :", F.relu(x))
print("nn.ReLU()(x) :", nn.ReLU()(x), " <- identical, just wrapped in a Module")
print()
print("MLP.forward (detr.py:298-301) picks per layer:")
print("   x = F.relu(layer(x)) if i < self.num_layers - 1 else layer(x)")
print(" -> ReLU between layers, but NOT after the last one, because the output")
print("    goes straight into .sigmoid() for the box coordinates.")

nn.Linear state : ['weight', 'bias']
F.relu state    : none -- it is just max(x, 0)

F.relu(x)    : tensor([0.000, 0.308, 0.000, 0.198])
nn.ReLU()(x) : tensor([0.000, 0.308, 0.000, 0.198])  <- identical, just wrapped in a Module

MLP.forward (detr.py:298-301) picks per layer:
   x = F.relu(layer(x)) if i < self.num_layers - 1 else layer(x)
 -> ReLU between layers, but NOT after the last one, because the output
    goes straight into .sigmoid() for the box coordinates.


### `F.interpolate`, and a dtype dance worth studying

[`backbone.py:78`](../models/backbone.py#L78) has to shrink the padding mask from image resolution down to feature-map resolution, so it still lines up after ResNet's 32× downsampling:

```python
mask = F.interpolate(m[None].float(), size=x.shape[-2:]).to(torch.bool)[0]
```

Four operations in one line, and every one is necessary:

In [52]:
m = torch.zeros(1, 800, 1066, dtype=torch.bool)     # (B, H, W)
m[:, :, 800:] = True                                # right side is padding
feat_hw = (25, 34)

print("1. m[None]          ", tuple(m[None].shape),
      "   interpolate needs a channel axis: (B,C,H,W)")
try:
    F.interpolate(m[None], size=feat_hw)
except NotImplementedError as e:
    print("2. .float() needed  -> without it:", str(e).splitlines()[0][:52])
print("3. size=x.shape[-2:]   match the feature grid exactly, not a scale factor")

out = F.interpolate(m[None].float(), size=feat_hw).to(torch.bool)[0]
print("4. .to(torch.bool)[0]", tuple(out.shape), "  back to a mask, channel axis dropped")
print()
print("result -- padding survives the downsample:")
print(out[0].int())

1. m[None]           (1, 1, 800, 1066)    interpolate needs a channel axis: (B,C,H,W)
2. .float() needed  -> without it: "upsample_nearest2d_channels_last" not implemented f
3. size=x.shape[-2:]   match the feature grid exactly, not a scale factor
4. .to(torch.bool)[0] (1, 25, 34)   back to a mask, channel axis dropped

result -- padding survives the downsample:
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

Default interpolation here is **nearest**, which is the right choice for a mask: bilinear would produce fractional values that mean nothing once cast back to `bool`. (Contrast [`segmentation.py`](../models/segmentation.py), which *does* use `mode="bilinear"` — because there it is upsampling continuous mask logits, not a binary flag.)

---

## 11. Losses eat logits, not probabilities

`F.cross_entropy` applies `log_softmax` internally. Feeding it softmax output is a real and common bug — it "works" (the loss decreases) but the gradients are wrong and training is much worse.

In [53]:
logits = torch.randn(4, 6)
target = torch.tensor([0, 3, 5, 5])

auto   = F.cross_entropy(logits, target)
manual = F.nll_loss(F.log_softmax(logits, dim=-1), target)
print("F.cross_entropy(logits)           :", round(auto.item(), 6))
print("F.nll_loss(F.log_softmax(logits)) :", round(manual.item(), 6))
print("identical:", torch.allclose(auto, manual))
print()
print("BUG: F.cross_entropy(softmax(logits)):",
      round(F.cross_entropy(F.softmax(logits, -1), target).item(), 6),
      " <- wrong, and silent")
print()
print("This is also why the model returns RAW logits (detr.py:67) and lets each")
print("consumer choose: cross_entropy wants logits, the matcher wants softmax,")
print("PostProcess wants softmax. One output, three correct uses.")

F.cross_entropy(logits)           : 3.329936
F.nll_loss(F.log_softmax(logits)) : 3.329936
identical: True

BUG: F.cross_entropy(softmax(logits)): 1.913368  <- wrong, and silent

This is also why the model returns RAW logits (detr.py:67) and lets each
consumer choose: cross_entropy wants logits, the matcher wants softmax,
PostProcess wants softmax. One output, three correct uses.


### The shape convention, and the silent failure inside it

[`detr.py:121`](../models/detr.py#L121):

```python
loss_ce = F.cross_entropy(src_logits.transpose(1, 2), target_classes, self.empty_weight)
```

That `transpose(1, 2)` is mandatory. For inputs with more than 2 dimensions, `cross_entropy` expects **`(N, C, d1, d2, ...)`** — the class axis must be axis **1**, not last. DETR's logits are `(B, queries, classes)`, so they are transposed to `(B, classes, queries)`.

In [54]:
B_, Q, NC = 2, 5, 6
src_logits = torch.randn(B_, Q, NC)
target_classes = torch.randint(0, NC, (B_, Q))
ew = torch.ones(NC); ew[-1] = 0.1

print("src_logits               ", tuple(src_logits.shape), " (B, queries, classes)")
print("src_logits.transpose(1,2)", tuple(src_logits.transpose(1, 2).shape), " (B, classes, queries)")
print("target_classes           ", tuple(target_classes.shape), target_classes.dtype)
print()
print("correct loss:", round(F.cross_entropy(src_logits.transpose(1, 2), target_classes, ew).item(), 4))

try:
    F.cross_entropy(src_logits, target_classes)
except RuntimeError as e:
    print("\nwithout the transpose -> RuntimeError:", str(e).splitlines()[0][:60])

src_logits                (2, 5, 6)  (B, queries, classes)
src_logits.transpose(1,2) (2, 6, 5)  (B, classes, queries)
target_classes            (2, 5) torch.int64

correct loss: 2.3066

without the transpose -> RuntimeError: Expected target size [2, 6], got [2, 5]


Usually it fails loudly, as above. But watch what happens when the number of queries happens to **equal** the number of classes — the shapes become self-consistent and PyTorch cannot tell that you meant something else:

In [55]:
sq = torch.randn(2, 6, 6)                 # queries == classes == 6
tg = torch.randint(0, 6, (2, 6))

print("with    transpose:", round(F.cross_entropy(sq.transpose(1, 2), tg).item(), 4), " correct")
print("without transpose:", round(F.cross_entropy(sq, tg).item(), 4), " <- runs, silently wrong")
print()
print("Same family of bug as the broadcasting diagonal in §3: when two axes")
print("coincidentally match, an axis mix-up stops being an error and becomes")
print("a wrong number. Always name your axes in a comment.")

with    transpose: 2.2135  correct
without transpose: 2.0637  <- runs, silently wrong

Same family of bug as the broadcasting diagonal in §3: when two axes
coincidentally match, an axis mix-up stops being an error and becomes
a wrong number. Always name your axes in a comment.


### `weight=` and the ∅ imbalance

`empty_weight` down-weights the no-object class by `eos_coef` (0.1). It is worth knowing exactly when that has an effect — and when it has none at all.

In [56]:
NC = 92
lg = torch.zeros(1, 100, NC); lg[..., -1] = 10.0          # confidently predict no-object
tg = torch.full((1, 100), 91, dtype=torch.int64)          # and every target IS no-object
w = torch.ones(NC); w[-1] = 0.1

print("all targets are no-object:")
print("  unweighted:", round(F.cross_entropy(lg.transpose(1, 2), tg).item(), 6))
print("  weighted  :", round(F.cross_entropy(lg.transpose(1, 2), tg, w).item(), 6),
      " <- identical!")
print("  (weight normalizes by the sum of the targets' weights, so a single")
print("   uniform class cancels out entirely.)")

tg[0, :10] = 17                                            # ten queries hold a real class
print("\nten real objects, badly predicted:")
print("  unweighted:", round(F.cross_entropy(lg.transpose(1, 2), tg).item(), 4))
print("  weighted  :", round(F.cross_entropy(lg.transpose(1, 2), tg, w).item(), 4),
      " <- much larger")
print()
print("That is what --eos_coef 0.1 buys: the 10 rare real objects are no longer")
print("drowned out by 90 easy negatives. Paper 3.1, 'Loss'.")

all targets are no-object:
  unweighted: 0.004123
  weighted  : 0.004123  <- identical!
  (weight normalizes by the sum of the targets' weights, so a single
   uniform class cancels out entirely.)

ten real objects, badly predicted:
  unweighted: 1.0041
  weighted  : 5.2673  <- much larger

That is what --eos_coef 0.1 buys: the 10 rare real objects are no longer
drowned out by 90 easy negatives. Paper 3.1, 'Loss'.


### `reduction='none'`

By default losses average over everything. DETR needs to normalize by the **number of ground-truth boxes across the batch**, not by the number of elements, so it asks for the raw per-element loss and divides itself ([`detr.py:153-156`](../models/detr.py#L153-L156)):

```python
loss_bbox = F.l1_loss(src_boxes, target_boxes, reduction='none')
losses['loss_bbox'] = loss_bbox.sum() / num_boxes
```

In [57]:
src_boxes = torch.rand(3, 4)         # 3 matched predictions
tgt_boxes = torch.rand(3, 4)
num_boxes = 3.0

per_elem = F.l1_loss(src_boxes, tgt_boxes, reduction='none')
print("reduction='none' ->", tuple(per_elem.shape), " one value per coordinate")
print("reduction='mean' ->", tuple(F.l1_loss(src_boxes, tgt_boxes).shape), " a scalar")
print()
print("DETR's normalization :", round((per_elem.sum() / num_boxes).item(), 4))
print("plain mean           :", round(F.l1_loss(src_boxes, tgt_boxes).item(), 4))
print()
print("A factor of 4 apart (coordinates per box) -- but that is not the point.")
print("The point is that num_boxes is summed ACROSS GPUS (detr.py:229-232),")
print("so the loss scale does not drift with how many objects a batch happens")
print("to contain, or with how many workers you train on.")

reduction='none' -> (3, 4)  one value per coordinate
reduction='mean' -> ()  a scalar

DETR's normalization : 1.3704
plain mean           : 0.3426

A factor of 4 apart (coordinates per box) -- but that is not the point.
The point is that num_boxes is summed ACROSS GPUS (detr.py:229-232),
so the loss scale does not drift with how many objects a batch happens
to contain, or with how many workers you train on.


### One more: `torch.diag` on a pairwise matrix

[`detr.py:158-160`](../models/detr.py#L158-L160) reuses the all-pairs GIoU function from §3 but only wants the **matched** pairs, so it takes the diagonal:

```python
loss_giou = 1 - torch.diag(box_ops.generalized_box_iou(
    box_ops.box_cxcywh_to_xyxy(src_boxes),
    box_ops.box_cxcywh_to_xyxy(target_boxes)))
```

Because `src_boxes` and `target_boxes` were gathered in the same order (§4), pair *i* is exactly row *i*, column *i*.

In [58]:
from util.box_ops import generalized_box_iou

src = torch.tensor([[0., 0., 10., 10.], [5., 5., 15., 15.]])
tgt = torch.tensor([[1., 1., 11., 11.], [5., 5., 16., 16.]])

full = generalized_box_iou(src, tgt)
print("full pairwise GIoU", tuple(full.shape), ":")
print(full)
print("\ntorch.diag(...)  ->", torch.diag(full), "  <- only the matched pairs")
print("loss_giou = 1 - diag ->", (1 - torch.diag(full)))
print()
print("Computing N*M values to use N of them is wasteful but negligible here,")
print("and it lets the loss reuse the exact function the matcher already needs.")

full pairwise GIoU (2, 2) :
tensor([[ 0.664, -0.107],
        [ 0.056,  0.826]])

torch.diag(...)  -> tensor([0.664, 0.826])   <- only the matched pairs
loss_giou = 1 - diag -> tensor([0.336, 0.174])

Computing N*M values to use N of them is wasteful but negligible here,
and it lets the loss reuse the exact function the matcher already needs.


---

## 12. `state_dict`: saving, loading, and the fine-tuning gotcha

A `state_dict` is an `OrderedDict` mapping **dotted attribute paths** to tensors. The paths come straight from how you named your attributes — which is why `DETR` in this repo loads the official Facebook checkpoint with zero key renaming.

In [59]:
sd = head.state_dict()          # the MLP bbox head
print("MLP state_dict:")
for k, v in sd.items():
    print(f"  {k:<20} {tuple(v.shape)}")
print()
print("'layers.0.weight' = attribute 'layers' -> ModuleList index 0 -> its 'weight'.")
print("Rename the attribute and every checkpoint key changes with it.")

MLP state_dict:
  layers.0.weight      (256, 256)
  layers.0.bias        (256,)
  layers.1.weight      (256, 256)
  layers.1.bias        (256,)
  layers.2.weight      (4, 256)
  layers.2.bias        (4,)

'layers.0.weight' = attribute 'layers' -> ModuleList index 0 -> its 'weight'.
Rename the attribute and every checkpoint key changes with it.


### `strict=False` does **not** mean "ignore everything"

[`tutorial/detr_utils.py:123-125`](detr_utils.py) loads the checkpoint and then asserts that nothing was missing or unexpected:

```python
ck = torch.hub.load_state_dict_from_url(DETR_R50_URL, map_location="cpu")
missing, unexpected = model.load_state_dict(ck["model"], strict=False)
assert not missing and not unexpected, (missing, unexpected)
```

`strict=False` tolerates **missing** and **unexpected keys**. It does *not* tolerate a **shape mismatch** — that still raises:

In [60]:
class Net(nn.Module):
    def __init__(self, n_out=92):
        super().__init__()
        self.backbone = nn.Linear(4, 4)
        self.class_embed = nn.Linear(4, n_out)

full = Net().state_dict()

# 1. missing key -- tolerated
partial = {k: v for k, v in full.items() if not k.startswith("class_embed")}
r = Net().load_state_dict(partial, strict=False)
print("1. missing keys   :", r.missing_keys)
print("   unexpected keys:", r.unexpected_keys)

# 2. unexpected key -- tolerated
extra = dict(full); extra["nonexistent.weight"] = torch.zeros(1)
r = Net().load_state_dict(extra, strict=False)
print("\n2. unexpected keys:", r.unexpected_keys)

# 3. SHAPE MISMATCH -- not tolerated, even with strict=False
try:
    Net(n_out=21).load_state_dict(full, strict=False)
except RuntimeError as e:
    print("\n3. shape mismatch -> RuntimeError:", str(e).splitlines()[0][:62])
    print("   ", [l.strip() for l in str(e).splitlines() if "class_embed" in l][:1])

1. missing keys   : ['class_embed.weight', 'class_embed.bias']
   unexpected keys: []

2. unexpected keys: ['nonexistent.weight']

3. shape mismatch -> RuntimeError: Error(s) in loading state_dict for Net:
    ['size mismatch for class_embed.weight: copying a param with shape torch.Size([92, 4]) from checkpoint, the shape in current model is torch.Size([21, 4]).']


**That third case is the fine-tuning gotcha.** The released checkpoint's `class_embed` is `Linear(256, 92)`, baked for COCO's 91 classes + ∅. Build DETR with a different `num_classes` and `strict=False` will not save you.

The fix is to drop the head's keys before loading and let it train from scratch, keeping the backbone and transformer — this is exactly what notebook [`07`](07_finetune_on_your_own_data.ipynb) does:

In [61]:
ck = Net().state_dict()                       # pretend this is the COCO checkpoint
model_new = Net(n_out=21)                     # a 20-class dataset + no-object

for k in [k for k in ck if k.startswith("class_embed")]:
    del ck[k]

missing, unexpected = model_new.load_state_dict(ck, strict=False)
print("after deleting the head:")
print("  missing   :", missing, " <- expected: the new head trains from scratch")
print("  unexpected:", unexpected)
print()
print("The backbone weights loaded fine. Only the classifier is fresh.")
print("See models/detr.py:304-312 for why num_classes means max_obj_id + 1.")

after deleting the head:
  missing   : ['class_embed.weight', 'class_embed.bias']  <- expected: the new head trains from scratch
  unexpected: []

The backbone weights loaded fine. Only the classifier is fresh.
See models/detr.py:304-312 for why num_classes means max_obj_id + 1.


One more detail from [`main.py:172-178`](../main.py#L172-L178): `--resume` accepts a **URL**, not just a path, and the checkpoint is a dict with several top-level keys, of which `"model"` is only one:

```python
checkpoint = torch.hub.load_state_dict_from_url(args.resume, map_location='cpu', check_hash=True)
model_without_ddp.load_state_dict(checkpoint['model'])
```

Training checkpoints also carry `'optimizer'`, `'lr_scheduler'`, `'epoch'` and `'args'`, so a run can resume exactly where it stopped. That is why the DETR-R50 file is 159 MB for a ~41 M parameter model — AdamW stores two extra moment tensors per parameter.

---

## 13. Leaving tensor-land

| Call | Gives you | Use when |
|---|---|---|
| `.item()` | a Python number | logging a scalar loss |
| `.tolist()` | nested Python lists | small results, printing |
| `.detach().cpu().numpy()` | a NumPy array | matplotlib, OpenCV |

The order matters: `.detach()` first (drop the graph), then `.cpu()` (leave the accelerator), then `.numpy()`. Calling `.numpy()` on a tensor that still requires grad, or that lives on CUDA/MPS, raises.

In [62]:
t = torch.randn(2, 3, requires_grad=True)

try:
    t.numpy()
except RuntimeError as e:
    print("t.numpy()  ->", str(e).splitlines()[0][:72])

print("t.detach().cpu().numpy() ->", type(t.detach().cpu().numpy()).__name__,
      t.detach().cpu().numpy().shape)
print("t.sum().item()           ->", round(t.sum().item(), 4))
print("t.tolist()               ->", [[round(v, 2) for v in r] for r in t.tolist()])
print()
print("Caveat: on GPU, .item() forces a host synchronization. Calling it every")
print("iteration inside a training loop is a classic silent slowdown -- which is")
print("why util/misc.py batches metric logging through SmoothedValue instead.")

t.numpy()  -> Can't call numpy() on Tensor that requires grad. Use tensor.detach().num
t.detach().cpu().numpy() -> ndarray (2, 3)
t.sum().item()           -> -0.7508
t.tolist()               -> [[-0.41, 0.02, -1.47], [0.11, 1.21, -0.22]]

Caveat: on GPU, .item() forces a host synchronization. Calling it every
iteration inside a training loop is a classic silent slowdown -- which is
why util/misc.py batches metric logging through SmoothedValue instead.


---

## 14. The whole forward pass, as shapes

Everything above, assembled. This is [`detr.py:59-72`](../models/detr.py#L59-L72) annotated — the shapes a single 800×1066 image produces.

In [63]:
rows = [
    ("input image (padded batch)",       "(1, 3, 800, 1066)", "NestedTensor.tensors", "1"),
    ("  + padding mask",                 "(1, 800, 1066)",    "dtype=bool",            "1"),
    ("ResNet-50 layer4",                 "(1, 2048, 25, 34)", "backbone, /32 stride",  "7"),
    ("mask, F.interpolate'd",            "(1, 25, 34)",       "nearest, bool round-trip", "10"),
    ("input_proj  Conv2d(1x1)",          "(1, 256, 25, 34)",  "2048 -> 256 channels",  "7"),
    ("flatten(2).permute(2,0,1)",        "(850, 1, 256)",     "SEQUENCE: (HW, B, C)",  "2"),
    ("position encoding, same treatment","(850, 1, 256)",     "added to Q and K only", "5"),
    ("mask.flatten(1)",                  "(1, 850)",          "key_padding_mask",      "7"),
    ("encoder memory",                   "(850, 1, 256)",     "self-attention x6",     "6"),
    ("query_embed.weight",               "(100, 256)",        "nn.Embedding, used raw","7"),
    ("  .unsqueeze(1).repeat(1,B,1)",    "(100, 1, 256)",     "same queries per image","3"),
    ("decoder hs (all 6 layers)",        "(6, 1, 100, 256)",  "after .transpose(1,2)", "2"),
    ("class_embed  Linear(256, 92)",     "(6, 1, 100, 92)",   "logits; last axis only","7"),
    ("bbox_embed   MLP -> .sigmoid()",   "(6, 1, 100, 4)",    "cxcywh in [0,1]",       "7"),
    ("pred_logits = outputs_class[-1]",  "(1, 100, 92)",      "final decoder layer",   "4"),
    ("softmax(-1)[..., :-1].max(-1)",    "(1, 100) x2",       "scores, labels",        "5"),
]
print(f"{'stage':<36}{'shape':<22}{'note':<26}§")
print("-" * 92)
for a, b_, c_, s_ in rows:
    print(f"{a:<36}{b_:<22}{c_:<26}{s_}")

stage                               shape                 note                      §
--------------------------------------------------------------------------------------------
input image (padded batch)          (1, 3, 800, 1066)     NestedTensor.tensors      1
  + padding mask                    (1, 800, 1066)        dtype=bool                1
ResNet-50 layer4                    (1, 2048, 25, 34)     backbone, /32 stride      7
mask, F.interpolate'd               (1, 25, 34)           nearest, bool round-trip  10
input_proj  Conv2d(1x1)             (1, 256, 25, 34)      2048 -> 256 channels      7
flatten(2).permute(2,0,1)           (850, 1, 256)         SEQUENCE: (HW, B, C)      2
position encoding, same treatment   (850, 1, 256)         added to Q and K only     5
mask.flatten(1)                     (1, 850)              key_padding_mask          7
encoder memory                      (850, 1, 256)         self-attention x6         6
query_embed.weight                  (100, 256)

Every shape change in that table is one of the four verbs from §2. Every `-1` is either "last decoder layer" or "drop the no-object class" (§4). Every layer with weights is one of the four from §7. That really is the whole vocabulary.

## What's next

- **Came here first?** Go to [`01_attention_from_scratch.ipynb`](01_attention_from_scratch.ipynb) and build attention out of the pieces above — §6 is the direct prerequisite.
- **Sent here from another notebook?** The section you needed is in the map at the top.
- **Want to see it all at once?** [`08_build_detr_from_scratch.ipynb`](08_build_detr_from_scratch.ipynb) writes the whole model in 50 lines using nothing but the operations in this notebook.

---

## Exercises

**Exercise 1.** `src.flatten(2).permute(2, 0, 1)` turns `(B,C,H,W)` into `(HW,B,C)`. Write the inverse — get back to `(B,C,H,W)` — and check it round-trips exactly.

<details><summary>Solution</summary>

```python
src = torch.randn(2, 256, 25, 34)
seq = src.flatten(2).permute(2, 0, 1)             # (850, 2, 256)
back = seq.permute(1, 2, 0).view(2, 256, 25, 34)  # exactly transformer.py:59
print(torch.equal(src, back))                     # True
```

`view` succeeds even though `seq.permute(1,2,0)` is not contiguous — splitting the last axis is stride-compatible. `.reshape(...)` also works but may copy.

A useful sanity check on the token ordering: token `i` corresponds to grid position `(i // W, i % W)`.

```python
i = 40
print(torch.equal(seq[i, 0], src[0, :, i // 34, i % 34]))   # True
```
</details>

---

**Exercise 2.** Why does `torch.max(boxes1[:, None, :2], boxes2[:, :2])` give an `(N, M, 2)` result? Work the broadcasting by hand for N=3, M=2, then predict what happens *without* the `None` when N ≠ M, and when N == M.

<details><summary>Solution</summary>

`boxes1[:, None, :2]` is `(3, 1, 2)`; `boxes2[:, :2]` is `(2, 2)`, padded to `(1, 2, 2)`. Aligning from the right: `2` vs `2` match; `1` vs `2` stretches to 2; `3` vs `1` stretches to 3. Result `(3, 2, 2)`.

Without the `None` and with N ≠ M you get a clean error:

```python
torch.max(torch.rand(3, 2), torch.rand(2, 2))
# RuntimeError: The size of tensor a (3) must match the size of tensor b (2)
```

That is the *lucky* case. When N == M it broadcasts fine and silently computes only the **diagonal** pairs — the matcher would then match every prediction to the target that happens to share its index, and training would quietly fail to converge. Same failure family as the `cross_entropy` transpose in §11.
</details>

---

**Exercise 3.** Show that a module kept in a plain Python list really does stay frozen during training, and that it vanishes from the checkpoint.

<details><summary>Solution</summary>

The pure-list version is caught by PyTorch — `torch.optim.SGD` raises `ValueError: optimizer got an empty parameter list`. The dangerous case is a *mix*: the optimizer is non-empty, training runs, and the unregistered layer never moves.

```python
class Mixed(nn.Module):
    def __init__(self):
        super().__init__()
        self.good = nn.Linear(4, 4)
        self.bad  = [nn.Linear(4, 4)]          # <- invisible
    def forward(self, x):
        return self.bad[0](self.good(x))

m = Mixed()
b_good, b_bad = m.good.weight.clone(), m.bad[0].weight.clone()
opt = torch.optim.SGD(m.parameters(), lr=0.1)
for _ in range(5):
    loss = m(torch.randn(8, 4)).pow(2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

print("good changed:", not torch.equal(b_good, m.good.weight))    # True
print("bad  changed:", not torch.equal(b_bad,  m.bad[0].weight))  # False  <- the bug
print("state_dict keys:", list(m.state_dict().keys()))            # only 'good.*'
```

The `state_dict` line is the real tell: save this model and the second layer is not in the checkpoint at all. Loading it back gives you a randomly initialized layer with no warning, because `strict=True` only checks keys that the *module* declares.
</details>

---

**Exercise 4.** Redo the §4 scatter with the real 100 queries and three images, then verify that exactly as many entries differ from 91 as there are ground-truth boxes.

<details><summary>Solution</summary>

```python
indices = [(torch.tensor([7, 42]),      torch.tensor([0, 1])),
           (torch.tensor([3]),          torch.tensor([0])),
           (torch.tensor([11, 60, 99]), torch.tensor([0, 1, 2]))]
labels  = [torch.tensor([17, 18]), torch.tensor([3]), torch.tensor([1, 1, 62])]

batch_idx = torch.cat([torch.full_like(s, i) for i, (s, _) in enumerate(indices)])
src_idx   = torch.cat([s for (s, _) in indices])

tc = torch.full((3, 100), 91, dtype=torch.int64)
tc[(batch_idx, src_idx)] = torch.cat([l[J] for l, (_, J) in zip(labels, indices)])

n_boxes = sum(len(l) for l in labels)
print((tc != 91).sum().item(), "==", n_boxes)      # 6 == 6
print("per image:", (tc != 91).sum(1).tolist())    # [2, 1, 3]
```

`(tc != 91).sum(1)` is precisely the cardinality metric from [`detr.py:138`](../models/detr.py#L138), computed on targets instead of predictions. Comparing the two is what `loss_cardinality` logs — if the model predicts 30 objects where the image has 3, that gap shows up here long before mAP does.
</details>

---

**Exercise 5.** `F.cross_entropy(..., weight=empty_weight)` down-weights ∅ by 10×. Find the case where the weight makes **no** difference at all, and explain why.

<details><summary>Solution</summary>

```python
NC = 92
logits = torch.zeros(1, 100, NC); logits[..., -1] = 10.0
target = torch.full((1, 100), 91, dtype=torch.int64)
w = torch.ones(NC); w[-1] = 0.1

print(F.cross_entropy(logits.transpose(1, 2), target).item())
print(F.cross_entropy(logits.transpose(1, 2), target, w).item())   # identical
```

With the default `reduction='mean'`, a weighted cross-entropy computes `sum(w_i * l_i) / sum(w_i)` — it normalizes by the **sum of the targets' weights**, not by the count. When every target has the same weight, the factor appears in numerator and denominator and cancels exactly.

The weight only bites in a mixture (`target[0, :10] = 17`), where the ten real objects keep weight 1.0 while the ninety ∅ queries contribute 0.1 each to the denominator. The effective share of the loss owned by real objects rises from 10% to roughly 53%.
</details>

---

**Exercise 6.** `nn.Linear` maps the last axis. Verify that `class_embed(hs)` on a 4-D `hs` is identical to flattening, applying the layer, and unflattening — and time both.

<details><summary>Solution</summary>

```python
import time
lin = nn.Linear(256, 92)
hs = torch.randn(6, 2, 100, 256)

direct = lin(hs)
manual = lin(hs.reshape(-1, 256)).reshape(6, 2, 100, 92)
print("identical:", torch.allclose(direct, manual, atol=1e-6))

for name, fn in [("direct", lambda: lin(hs)),
                 ("manual", lambda: lin(hs.reshape(-1, 256)).reshape(6, 2, 100, 92))]:
    t0 = time.perf_counter()
    for _ in range(100): fn()
    print(f"{name}: {(time.perf_counter()-t0)*10:.2f} ms / call")
```

They are the same computation — `Linear` flattens internally. The direct form is not meaningfully faster; it is just impossible to get the unflattening wrong.
</details>

---

**Exercise 7.** Reconstruct DETR's full output dict from raw pieces, using only operations from this notebook — no model. Start from a random `hs` of shape `(6, 1, 100, 256)` and produce `pred_logits`, `pred_boxes` and `aux_outputs` matching [`detr.py:67-72`](../models/detr.py#L67-L72).

<details><summary>Solution</summary>

```python
from models.detr import MLP

hs = torch.randn(6, 1, 100, 256)
class_embed = nn.Linear(256, 92)
bbox_embed  = MLP(256, 256, 4, 3)

outputs_class = class_embed(hs)                       # (6, 1, 100, 92)
outputs_coord = bbox_embed(hs).sigmoid()              # (6, 1, 100, 4)

out = {'pred_logits': outputs_class[-1], 'pred_boxes': outputs_coord[-1]}
out['aux_outputs'] = [{'pred_logits': a, 'pred_boxes': b}
                      for a, b in zip(outputs_class[:-1], outputs_coord[:-1])]

print({k: (tuple(v.shape) if torch.is_tensor(v) else f"list of {len(v)}")
       for k, v in out.items()})
# {'pred_logits': (1, 100, 92), 'pred_boxes': (1, 100, 4), 'aux_outputs': 'list of 5'}

assert (outputs_coord >= 0).all() and (outputs_coord <= 1).all()   # sigmoid guarantees it
```

Then run it through `PostProcess`:

```python
from models.detr import PostProcess
res = PostProcess()(out, torch.tensor([[800, 1066]]))[0]
print({k: tuple(v.shape) for k, v in res.items()})
# {'scores': (100,), 'labels': (100,), 'boxes': (100, 4)}
```

Six `zip`s, two heads, one `sigmoid`, one `[-1]`. That is the entire output side of DETR.
</details>